# LocateAnything-3B → ONNX → TensorRT (one-click autopilot)

**Model:** [`nvidia/LocateAnything-3B`](https://huggingface.co/nvidia/LocateAnything-3B) — native-resolution Vision-Language Grounding with **Parallel Box Decoding (PBD)**.

**Recommended runtime:** Colab L4 (24 GB) or A100. Free T4 will finish vision + projector, but the LLM engine build is skipped (insufficient VRAM).

Every cell is **idempotent** and **side-effect safe** — re-running picks up cached artifacts. Failures in optional cells (LLM engine on T4, video download, PyTorch benchmark) do not kill the run.

## Architecture map

```
                pixel_values (1,3,H,W)            input_ids (1,S)
                        │                                │
        ┌───────────────▼────────────────┐                │
        │ MoonViT-SO-400M (vision_model) │                │
        │   patch=14 · 27L · d=1152 · 16H │                │
        │   2-D RoPE  (complex→real)      │                │
        │   flash_attn_varlen → SDPA      │                │
        │   2×2 patch merger              │                │
        └───────────────┬────────────────┘                │
                        │ (L_post, 4608)                   │
        ┌───────────────▼────────────────┐                │
        │ mlp1: LN→Lin 4608→2048→GELU→    │                │
        │           Lin 2048→2048         │                │
        └───────────────┬────────────────┘                │
                        │ scatter @image_token_index      │
                        ▼                                ▼
        ┌──────────────────────────────────────────────────┐
        │ Qwen2.5-3B-Instruct  (36L·d2048·GQA 16/2·vocab 152681) │
        │ RoPE θ=1e6 · ctx 32K · PBD block_size=6 (MTP head)     │
        └───────────────┬──────────────────────────────────┘
                        │ logits (1,K,V)
                        ▼
        ┌──────────────────────────────────────────────┐
        │ PBD orchestrator (Python)                    │
        │   MTP-fast │ MTP+AR hybrid │ pure AR slow     │
        │   parse <box><x><y><x><y></box> tokens        │
        └──────────────────────────────────────────────┘
```

**Optional: §7b runs INT4 AWQ quantisation via NVIDIA Model Optimizer so the LLM ONNX fits in the 2 GB Protobuf cap as a single file.**

**Export blockers fixed in §3:** `flash_attn_varlen_func`, complex `view_as_complex` RoPE, the custom `'magi'` attention implementation, variable-resolution input, dynamic MTP block size, externalised KV cache.

## 1 · Environment, GPU, and one-shot dependency install

In [ ]:
import subprocess, sys, os, time, json, gc, shutil, importlib, urllib.request
from pathlib import Path

def sh(cmd, check=True, quiet=False):
    if not quiet: print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout and not quiet: print(r.stdout[-3000:])
    if r.returncode and check:
        print(r.stderr[-3000:]); raise SystemExit(r.returncode)
    return r

try:
    import google.colab  # type: ignore  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
WORK = (Path('/content/locany') if ON_COLAB else (Path.home()/'locany'))
WORK.mkdir(parents=True, exist_ok=True)
ONNX_DIR = WORK/'onnx';     ONNX_DIR.mkdir(exist_ok=True)
TRT_DIR  = WORK/'engines';  TRT_DIR.mkdir(exist_ok=True)
print('Workdir:', WORK)

sh('nvidia-smi -L', check=False)
import torch
if not torch.cuda.is_available():
    msg = ('No CUDA device detected. This notebook requires an NVIDIA GPU runtime. '
           'In Colab: Runtime > Change runtime type > Hardware accelerator > GPU '
           '(L4 or A100 recommended; T4 finishes vision + projector only). '
           'Locally: ensure a CUDA-enabled PyTorch build and NVIDIA driver are installed.')
    raise SystemExit(msg)
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1024**3
ENABLE_LLM_TRT = VRAM_GB >= 22.0   # need ~12 GB headroom to build the LLM engine
print(f'GPU={GPU_NAME}  VRAM={VRAM_GB:.1f} GB  torch={torch.__version__}  ENABLE_LLM_TRT={ENABLE_LLM_TRT}')

In [ ]:
def have(mod):
    try:
        importlib.import_module(mod); return True
    except Exception: return False

# Only install what we need; pin nothing we don't have to, but keep ranges tight enough to be reproducible.
pkgs = []
if not have('transformers'):       pkgs.append('"transformers>=4.46,<4.55"')
if not have('accelerate'):         pkgs.append('accelerate')
if not have('safetensors'):        pkgs.append('safetensors')
if not have('einops'):             pkgs.append('einops')
if not have('sentencepiece'):      pkgs.append('sentencepiece')
if not have('onnx'):               pkgs.append('onnx')
# onnxruntime-gpu and onnxruntime share the `onnxruntime` import name but are
# different PyPI dists. Colab pre-installs CPU `onnxruntime`, so a `have('onnxruntime')`
# check would skip the GPU build and silently fall back to CPU in cell 25 (parity).
# Check for the CUDA provider, not the module, and evict any CPU wheel first.
try:
    import onnxruntime as _ort_probe
    _ort_has_cuda = any(p == 'CUDAExecutionProvider' for p in _ort_probe.get_available_providers())
except Exception:
    _ort_has_cuda = False
if not _ort_has_cuda:
    sh('pip -q uninstall -y onnxruntime onnxruntime-gpu', check=False)
    pkgs.append('onnxruntime-gpu')
if not have('tensorrt'):
    # Pin to TRT 10.7 cu12 line. Unpinned 'tensorrt' on PyPI currently resolves to
    # 11.x with tensorrt-cu13 deps and fails to load against Colab's CUDA-12 driver
    # ('libnvinfer.so.* cannot open shared object file').
    pkgs.append('tensorrt==10.7.*')
    pkgs.append('tensorrt-cu12==10.7.*')
if not have('cuda.bindings'):      pkgs.append('cuda-python>=12.6,<13')
if not have('cv2'):
    # Avoid mixed opencv-python + opencv-python-headless installs: same import name,
    # conflicting native libs, downstream VideoWriter/imshow can fail mysteriously.
    sh('pip -q uninstall -y opencv-python opencv-python-headless', check=False)
    pkgs.append('opencv-python-headless==4.10.*')
if not have('peft'):              pkgs.append('peft>=0.10,<0.13')
if not have('decord'):            pkgs.append('decord')
if not have('lmdb'):              pkgs.append('lmdb')
if not have('timm'):              pkgs.append('timm')
if not have('huggingface_hub'):   pkgs.append('huggingface_hub>=0.24,<1.0')
if not have('onnxscript'):        pkgs.append('onnxscript')
if not have('polygraphy'):        pkgs.append('polygraphy')
# Numpy 2.x breaks tensorrt-10.7 / onnxruntime-gpu / opencv ABI on Colab; pin to 1.x.
try:
    import numpy as _np_probe
    if int(_np_probe.__version__.split('.')[0]) >= 2:
        sh('pip -q install "numpy<2.0" --force-reinstall', check=False)
        # numpy is C-extension — needs a kernel restart to fully take effect.
        print('NOTE: numpy downgraded to <2.0 — if the next cell errors with a numpy ABI message, Runtime > Restart and re-run.')
except Exception:
    pkgs.append('numpy<2.0')
if pkgs:
    sh('pip -q install --no-input ' + ' '.join(pkgs))
    importlib.invalidate_caches()
    # Purge previously-imported entries so a re-run of cell 3 after a version change
    # actually picks up the new wheel without a kernel restart where possible.
    _stale = [k for k in list(sys.modules) if k.split('.')[0] in {'transformers','onnxruntime','tensorrt','cuda','peft'}]
    for k in _stale: sys.modules.pop(k, None)
    if _stale:
        print('NOTE: purged', len(_stale), 'cached modules. If imports below still fail, Runtime > Restart and re-run.')

for m in ['torch','transformers','onnx','onnxruntime','tensorrt']:
    print(f'{m:14s}', importlib.import_module(m).__version__)
# Hard-assert the exact submodule cell 32 needs (top-level `cuda` is a backcompat shim and may be a stale namespace package).
from cuda.bindings import runtime as _cudart_check  # noqa: F401
print(f'{"cuda.bindings":14s} OK')

In [ ]:
import numpy as np
import torch.nn.functional as F
import tensorrt as trt
import onnx, onnxruntime as ort
from huggingface_hub import snapshot_download
from PIL import Image

TRT_LOGGER = trt.Logger(trt.Logger.INFO)  # bumped from WARNING so build-phase errors surface (e.g. OOM details, unsupported op specifics)
MODEL_ID = 'nvidia/LocateAnything-3B'

# Show ONNX opset support of the local TRT (sanity).
print('TRT', trt.__version__, ' ONNX', onnx.__version__, ' ORT', ort.__version__)
_provs = ort.get_available_providers()
print('CUDA providers:', [p for p in _provs if 'CUDA' in p or 'Tensorrt' in p])
assert 'CUDAExecutionProvider' in _provs, (
    'onnxruntime-gpu not active — CUDAExecutionProvider missing. '
    'Re-run cell 3 (which now evicts CPU onnxruntime) and restart the runtime.')

## 2 · Download canonical weights (cached)

In [ ]:
LOCAL = Path(snapshot_download(
    MODEL_ID,
    local_dir=str(WORK/'weights'),
    allow_patterns=['*.json','*.py','*.txt','*.safetensors','*.safetensors.index.json','tokenizer*','chat_template*','generation_config*'],
))
print('Snapshot at:', LOCAL)
for f in sorted(p for p in LOCAL.rglob('*') if p.is_file() and not any(part.startswith('.') for part in p.relative_to(LOCAL).parts)):
    sz = f.stat().st_size / 1e6
    print(f'  {f.relative_to(LOCAL)}  ({sz:,.1f} MB)')

sys.path.insert(0, str(LOCAL))  # so the custom modeling_*.py becomes importable

## 3 · Define patches for ONNX-hostile ops

Three surgical patches that preserve numerical behaviour but make the graph traceable:

1. **`flash_attn_varlen_func` / `sdpa_attention` → SDPA** (single-image; multi-image production path uses the plugin in §10).
2. **2-D RoPE `view_as_complex` → real-valued `(cos, sin)`** via `Rope2DReal`.
3. **Force the `'magi'` attention implementation to `'sdpa'`** in both vision and LM configs.

Because the canonical `modeling_locateanything.py` uses **relative imports** (`from .configuration_locateanything import …`), we can't `import modeling_locateanything` directly — Python rejects relative imports outside a package context.  Instead this section just **defines** the patch functions; §4b looks the modules up in `sys.modules` after `AutoModel.from_pretrained` has loaded them under `transformers_modules.*` and applies the patches there.  That also guarantees we patch the same module objects the running model is actually using.

In [ ]:
# The custom modeling files in the HF snapshot use **relative imports**
# (e.g. `from .configuration_locateanything import …`).  Importing them as
# top-level modules fails with `attempted relative import with no known parent package`.
# transformers' `trust_remote_code=True` loader handles the package context
# correctly via `transformers_modules.<hash>.<module>`, so we let it do the
# load and then look the modules up in `sys.modules` AFTER `AutoModel.from_pretrained`
# returns.  This cell just defines the patch functions; §4b applies them.

import torch, sys, torch.nn.functional as F

def sdpa_packed(q, k, v, q_cu_seqlens=None, k_cu_seqlens=None):
    """Replacement for flash_attn_varlen_func / sdpa_attention.  Single-image path:
    no per-sample masking required because the orchestrator feeds one image per call.
    Production multi-image batching uses the plugin from §10."""
    q_ = q.transpose(0, 1).unsqueeze(0)
    k_ = k.transpose(0, 1).unsqueeze(0)
    v_ = v.transpose(0, 1).unsqueeze(0)
    out = F.scaled_dot_product_attention(q_, k_, v_, is_causal=False)
    # Match canonical contract: flatten last two dims (L, H, D) -> (L, H*D)
    # so the downstream wo Linear(H*D, H*D) sees the expected inner dim.
    return out.squeeze(0).transpose(0, 1).flatten(start_dim=-2).contiguous()

def apply_rope_real(xq, xk, freqs):
    """Real-valued 2-D RoPE.  Accepts either complex (canonical) or packed-real freqs.
    Mirrors the canonical apply_rope: inserts a singleton at dim -2 so freqs of shape
    (..., L, head_dim/2) broadcasts against xq/xk of shape (..., L, num_heads, head_dim/2).
    Without this unsqueeze the multiply fails: H (num_heads) collides with L on right-aligned broadcast."""
    if torch.is_complex(freqs):
        freqs = torch.stack([freqs.real, freqs.imag], dim=-1)
    cos = freqs[..., 0].unsqueeze(-2)   # (..., L, 1, head_dim/2)
    sin = freqs[..., 1].unsqueeze(-2)
    def rot(x):
        xp = x.float().reshape(*x.shape[:-1], -1, 2)
        xr, xi = xp.unbind(-1)
        out_r = xr * cos - xi * sin
        out_i = xr * sin + xi * cos
        return torch.stack([out_r, out_i], dim=-1).flatten(-2).to(x.dtype)
    return rot(xq), rot(xk)

class Rope2DReal(torch.nn.Module):
    """Drop-in replacement for the canonical Rope2DPosEmb that stores precomputed
    cos/sin buffers instead of a complex tensor, so the ONNX graph never touches
    `view_as_complex` (unsupported by every ONNX opset)."""
    def __init__(self, orig):
        super().__init__()
        self.dim = orig.dim
        f = orig.freqs_cis  # complex (H_max, W_max, dim/2)
        if f is None:
            # Canonical Rope2DPosEmb lazy-inits freqs_cis on first forward. Force it now.
            _dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            f = orig._precompute_freqs_cis(_dev)
            orig.freqs_cis = f
        self.register_buffer('freqs_cos', f.real.contiguous())
        self.register_buffer('freqs_sin', f.imag.contiguous())
    def get_freqs_cis(self, grid_hws):
        # Single-image: slice with the (h, w) carried by grid_hws[0]. Convert to Python ints
        # so (a) ONNX trace sees a shape-only slice (not a Range(Cast(Gather(grid_hws))) chain
        # whose shape is data-dependent on grid_hws values), and (b) we avoid int32-vs-int64
        # index_select issues on CUDA when grid_hws is int32.
        h_i = int(grid_hws[0, 0].item()) if torch.is_tensor(grid_hws[0, 0]) else int(grid_hws[0, 0])
        w_i = int(grid_hws[0, 1].item()) if torch.is_tensor(grid_hws[0, 1]) else int(grid_hws[0, 1])
        H_max, W_max = self.freqs_cos.shape[0], self.freqs_cos.shape[1]
        assert 1 <= h_i <= H_max and 1 <= w_i <= W_max, (h_i, w_i, H_max, W_max)
        cos = self.freqs_cos[:h_i, :w_i]
        sin = self.freqs_sin[:h_i, :w_i]
        cos = cos.reshape(-1, self.dim // 2)
        sin = sin.reshape(-1, self.dim // 2)
        # NOTE: real-stacked (L, dim/2, 2) output is ONLY valid in combination with the
        # apply_rope_real rebind from cell 13. Canonical apply_rope asserts dtype==complex64.
        return torch.stack([cos, sin], dim=-1)

# Ensure the model is built on the sdpa path (not 'magi') even before the patches land —
# AutoModel reads `_attn_implementation` from the config when constructing layers.
# We set it explicitly in §4; here we also disable flash-attn at the env level to avoid
# any sneak import in modeling_vit that would later try to call a None.
import os
os.environ['TRANSFORMERS_NO_FLASH_ATTN'] = '1'
print('Patch helpers defined.  Will be applied in §4b after AutoModel loads.')


## 4 · Load canonical model for parity baseline

In [ ]:
from transformers import AutoTokenizer, AutoProcessor, AutoModel, AutoConfig

tokenizer = AutoTokenizer.from_pretrained(str(LOCAL), trust_remote_code=True)
# use_fast=False keeps the vendored slow ImageProcessor that ships with the model (it has no fast variant; the
# transformers auto-upgrade just falls back anyway, but passing the flag silences the breaking-change warning).
processor = AutoProcessor.from_pretrained(str(LOCAL), trust_remote_code=True, use_fast=False)
config    = AutoConfig.from_pretrained(str(LOCAL), trust_remote_code=True)

# ---------------------------------------------------------------------------
# Backward-compat shim: transformers >=4.51 moved Qwen2's RoPE params into a
# `rope_parameters` dict, and the model's vendored modeling_qwen2.py still does
# direct attribute reads (`config.rope_theta`, `config.sliding_window`, ...).
# Rehydrate every text_config field from the original config.json so the
# constructor finds every attribute it expects.
# ---------------------------------------------------------------------------
with open(Path(LOCAL)/'config.json') as _f:
    _raw = json.load(_f)
for _k, _v in _raw.get('text_config', {}).items():
    setattr(config.text_config, _k, _v)
for _k, _v in _raw.get('vision_config', {}).items():
    setattr(config.vision_config, _k, _v)
# Sanity-check the attribute that triggered the original AttributeError.
assert hasattr(config.text_config, 'rope_theta'), 'rope_theta rehydration failed'
print(f'config.text_config: rope_theta={config.text_config.rope_theta} sliding={config.text_config.use_sliding_window} block_size={config.text_config.block_size}')

config._attn_implementation = 'sdpa'
config.text_config._attn_implementation = 'sdpa'
config.text_config.use_cache = True

# ---------------------------------------------------------------------------
# Backward-compat shim #2: transformers >=4.55 changed _tied_weights_keys from
# `List[str]` (legacy) to `Dict[str, str]` (target -> source) and added a
# post_init step that calls `.keys()` on it.  The vendored modeling_qwen2.py
# still uses the list form, so we upcast on the fly inside PreTrainedModel.
# ---------------------------------------------------------------------------
from transformers import modeling_utils as _mu
if not getattr(_mu.PreTrainedModel.get_expanded_tied_weights_keys, '_locany_patched', False):
    _orig_tied = _mu.PreTrainedModel.get_expanded_tied_weights_keys
    def _patched_tied(self, all_submodels=False):
        tk = getattr(self, '_tied_weights_keys', None)
        if isinstance(tk, (list, tuple, set)):
            # Upcast legacy list -> dict.  For tied embeddings each entry is the
            # *target* and it maps back to itself (transformers only checks shape
            # afterwards; it doesn't actually use the source path during load).
            self._tied_weights_keys = {k: k for k in tk}
        elif tk is None:
            self._tied_weights_keys = {}
        return _orig_tied(self, all_submodels)
    _patched_tied._locany_patched = True
    _mu.PreTrainedModel.get_expanded_tied_weights_keys = _patched_tied
    print('Patched PreTrainedModel.get_expanded_tied_weights_keys for legacy list _tied_weights_keys.')

# The outer LocateAnythingForConditionalGeneration.__init__ never calls self.post_init(),
# so `all_tied_weights_keys` is missing on the outer model when transformers'
# _finalize_load_state_dict subsequently calls mark_tied_weights_as_initialized.
# Patch it to no-op gracefully when the attribute is absent or empty.
if not getattr(_mu.PreTrainedModel.mark_tied_weights_as_initialized, '_locany_patched', False):
    _orig_mark = _mu.PreTrainedModel.mark_tied_weights_as_initialized
    def _patched_mark(self):
        tk = getattr(self, 'all_tied_weights_keys', None)
        if not tk:
            self.all_tied_weights_keys = {}
            return
        return _orig_mark(self)
    _patched_mark._locany_patched = True
    _mu.PreTrainedModel.mark_tied_weights_as_initialized = _patched_mark
    print('Patched PreTrainedModel.mark_tied_weights_as_initialized for models that skip post_init().')

# transformers >=4.56 removed DynamicCache.to_legacy_cache() and refactored the
# internal storage from key_cache/value_cache lists into a `layers` list of
# DynamicLayer objects.  The vendored modeling_qwen2.py still calls
# `next_decoder_cache.to_legacy_cache()`.  Restore the method with a body that
# handles both the new (`layers`) and old (`key_cache`/`value_cache`) layouts.
from transformers.cache_utils import DynamicCache as _DC
if not hasattr(_DC, 'to_legacy_cache') or getattr(_DC.to_legacy_cache, '_locany_shim', False) is False:
    def _to_legacy_cache(self):
        legacy = ()
        if hasattr(self, 'layers') and self.layers:
            for layer in self.layers:
                k = getattr(layer, 'keys', None);   v = getattr(layer, 'values', None)
                if k is None: k = getattr(layer, 'key_cache', None)
                if v is None: v = getattr(layer, 'value_cache', None)
                if k is not None and v is not None:
                    legacy += ((k, v),)
        elif hasattr(self, 'key_cache') and hasattr(self, 'value_cache'):
            for i in range(len(self.key_cache)):
                legacy += ((self.key_cache[i], self.value_cache[i]),)
        return legacy
    _to_legacy_cache._locany_shim = True
    _DC.to_legacy_cache = _to_legacy_cache
    # Round-trip: some code paths also expect from_legacy_cache to round-trip a tuple back into a DynamicCache.
    if not hasattr(_DC, 'from_legacy_cache'):
        @classmethod
        def _from_legacy_cache(cls, past):
            obj = cls()
            if past is None: return obj
            for k, v in past:
                obj.update(k, v, layer_idx=len(getattr(obj,'layers',[]) or getattr(obj,'key_cache',[])))
            return obj
        _DC.from_legacy_cache = _from_legacy_cache
    print('Patched DynamicCache.to_legacy_cache (and from_legacy_cache).')

REF_DTYPE = torch.float16
print('Loading 3B model ...')
model = AutoModel.from_pretrained(
    str(LOCAL), trust_remote_code=True, torch_dtype=REF_DTYPE, config=config,
).eval().to('cuda')
n_params = sum(p.numel() for p in model.parameters())
print(f'  loaded: {n_params/1e9:.2f} B params  (vision={sum(p.numel() for p in model.vision_model.parameters())/1e6:.0f} M, '
      f'projector={sum(p.numel() for p in model.mlp1.parameters())/1e6:.1f} M, '
      f'lm={sum(p.numel() for p in model.language_model.parameters())/1e9:.2f} B)'
)


In [ ]:
# ---- Reference forward + generation on a known image ----
import io, urllib.request
DEMO_IMG_URL = 'http://images.cocodataset.org/val2017/000000039769.jpg'
demo_path = WORK/'demo.jpg'
if not demo_path.exists():
    urllib.request.urlretrieve(DEMO_IMG_URL, str(demo_path))
demo_img = Image.open(demo_path).convert('RGB')
demo_img.thumbnail((672, 672))
print('demo image:', demo_img.size)

messages = [{'role':'user','content':[
    {'type':'image','image':demo_img},
    {'type':'text','text':'Detect all cats. Return bounding boxes.'},
]}]
text = processor.py_apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
images, videos = processor.process_vision_info(messages)
inputs = processor(text=[text], images=images, videos=videos, return_tensors='pt').to('cuda')
inputs['pixel_values'] = inputs['pixel_values'].to(REF_DTYPE)
# Normalise image_grid_hws to a torch int32 tensor.  The custom processor sometimes
# leaves it as a numpy array whose .dtype is `numpy.dtypes.Int64DType`, which the
# vendored modeling_vit.forward feeds into torch.zeros(..., dtype=...) and crashes.
_gh = inputs.get('image_grid_hws')
if _gh is not None and not torch.is_tensor(_gh):
    inputs['image_grid_hws'] = torch.as_tensor(_gh, dtype=torch.int32, device=inputs['pixel_values'].device)
elif torch.is_tensor(_gh):
    inputs['image_grid_hws'] = _gh.to(dtype=torch.int32)
print({k: tuple(v.shape) for k,v in inputs.items() if torch.is_tensor(v)})

with torch.inference_mode():
    ref_ids = model.generate(
        pixel_values=inputs['pixel_values'], input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'], image_grid_hws=inputs.get('image_grid_hws'),
        tokenizer=tokenizer, use_cache=True, max_new_tokens=128,
        generation_mode='hybrid', temperature=0.0, top_p=1.0, repetition_penalty=1.0,
        verbose=False,
    )
# generate() can return either token tensor or (tensor, info_tuple)
ref_tensor = ref_ids[0] if isinstance(ref_ids, tuple) else ref_ids
if isinstance(ref_tensor, torch.Tensor):
    new_ids = ref_tensor[0, inputs['input_ids'].shape[1]:] if ref_tensor.dim()==2 else ref_tensor
    REF_ANSWER = tokenizer.decode(new_ids, skip_special_tokens=False)
else:
    REF_ANSWER = str(ref_tensor)
print('\n=== REFERENCE ANSWER ===\n', REF_ANSWER[:600])

## 4b · Apply ONNX-hostile-op patches to the loaded modules

Now that `AutoModel.from_pretrained` has loaded `modeling_vit`, `modeling_qwen2`, and `modeling_locateanything` under `transformers_modules.*`, we look them up in `sys.modules` via the model's classes and rebind the three problematic globals (`multihead_attention`, `sdpa_attention`, `apply_rope`).  We also swap the live `Rope2DPosEmb` instance on the vision tower with `Rope2DReal` so the complex `freqs_cis` buffer never enters the ONNX graph.

In [ ]:
# Resolve the actual module objects transformers loaded for the canonical code.
mla_mod  = sys.modules[type(model).__module__]
mvit_mod = sys.modules[type(model.vision_model).__module__]
lm_cls   = type(model.language_model)
mqw_mod  = sys.modules[lm_cls.__module__]
print('Patching modules:')
print('  modeling_locateanything ->', mla_mod.__name__)
print('  modeling_vit            ->', mvit_mod.__name__)
print('  modeling_qwen2          ->', mqw_mod.__name__)

# --- (1) Vision attention: replace any flash/sdpa/eager packed-varlen path with sdpa_packed ---
for nm in ('multihead_attention', 'sdpa_attention', 'eager_attention', 'flash_attn_varlen_func'):
    if hasattr(mvit_mod, nm):
        setattr(mvit_mod, nm, sdpa_packed)
if hasattr(mvit_mod, 'VL_VISION_ATTENTION_FUNCTIONS'):
    for k in list(mvit_mod.VL_VISION_ATTENTION_FUNCTIONS.keys()):
        mvit_mod.VL_VISION_ATTENTION_FUNCTIONS[k] = sdpa_packed

# --- (2) Real-valued 2-D RoPE: rebind module-level apply_rope ---
mvit_mod.apply_rope = apply_rope_real
# Also rebind on any class that captured apply_rope into its own __dict__ at import time.
for _name in dir(mvit_mod):
    obj = getattr(mvit_mod, _name)
    if isinstance(obj, type) and 'apply_rope' in getattr(obj, '__dict__', {}):
        setattr(obj, 'apply_rope', apply_rope_real)

# --- (3) Swap the rope_2d instance on the vision tower so the graph sees real cos/sin buffers ---
vm = model.vision_model
swap_target = None
for nm in ('rope_2d', 'rope2d', 'pos_emb_2d', 'rope_emb'):
    if hasattr(vm, nm):
        swap_target = (vm, nm, getattr(vm, nm)); break
if swap_target is None:
    for sub_name, sub in vm.named_modules():
        if type(sub).__name__.startswith('Rope2D') and hasattr(sub, 'freqs_cis'):
            parts = sub_name.split('.')
            parent = vm
            for p in parts[:-1]: parent = getattr(parent, p)
            swap_target = (parent, parts[-1], sub); break
if swap_target is not None:
    parent, leaf, old = swap_target
    new = Rope2DReal(old).to(device=old.freqs_cis.device)
    setattr(parent, leaf, new)
    print(f'  swapped rope: {type(old).__name__} -> Rope2DReal  (path: .{leaf})')
else:
    print('  WARN: no Rope2DPosEmb instance found — relying on apply_rope_real complex->real fallback')

# --- (4) Belt-and-braces: if config still claims magi attention anywhere, neutralise it ---
if getattr(config, '_attn_implementation', None) == 'magi':
    config._attn_implementation = 'sdpa'
if hasattr(config, 'text_config') and getattr(config.text_config, '_attn_implementation', None) == 'magi':
    config.text_config._attn_implementation = 'sdpa'
if hasattr(mqw_mod, 'QWEN2_ATTENTION_CLASSES'):
    fallback = mqw_mod.QWEN2_ATTENTION_CLASSES.get('sdpa') or next(iter(mqw_mod.QWEN2_ATTENTION_CLASSES.values()))
    mqw_mod.QWEN2_ATTENTION_CLASSES['magi'] = fallback

# --- Smoke test the patched forward so we fail fast here, not 30 seconds into the ONNX trace ---
_gh = inputs['image_grid_hws']
if not torch.is_tensor(_gh):
    _gh = torch.as_tensor(_gh, dtype=torch.int32, device=inputs['pixel_values'].device)
else:
    _gh = _gh.to(dtype=torch.int32)
with torch.inference_mode():
    _ = model.vision_model(pixel_values=inputs['pixel_values'], grid_hws=_gh)
print('  patched vision forward OK')


## 5 · Export sub-graph 1: MoonViT vision encoder

In [ ]:
class VisionForExport(torch.nn.Module):
    """ONNX-traceable single-image vision encoder.

    The canonical MoonViT forward calls:
      - Learnable2DInterpPosEmb.forward    `for shape in grid_hws.tolist(): ...`
      - patch_merger                       `for x_shape in grid_hws.tolist(): ...`  + returns List[Tensor]
    Both materialise tensor data into Python lists, so the trace depends on the
    captured value rather than the shape. We work around both:
      (a) Inline patch_embed: pre-compute the interpolated pos_emb at __init__
          time (for one resolution) and register it as a buffer.  No loop in fwd.
      (b) Omit patch_merger from the graph — the orchestrator does it numerically
          after the engine returns (pure reshape+permute, no learned weights).

    To export for a different image resolution, re-instantiate VisionForExport
    with the new (grid_h, grid_w) and re-run the export cell.
    """
    def __init__(self, vit, grid_h, grid_w):
        super().__init__()
        self.patch_proj = vit.patch_embed.proj
        self.encoder    = vit.encoder
        weight     = vit.patch_embed.pos_emb.weight              # (H_max, W_max, d_model)
        interp_mode = vit.patch_embed.pos_emb.interpolation_mode
        w4 = weight.permute(2, 0, 1).unsqueeze(0)                # (1, d_model, H_max, W_max)
        ikw = {'mode': interp_mode}
        if interp_mode in ('bilinear', 'bicubic'): ikw['align_corners'] = False
        with torch.no_grad():
            pos = F.interpolate(w4, size=(int(grid_h), int(grid_w)), **ikw)
            pos = pos.squeeze(0).permute(1, 2, 0).flatten(end_dim=1).contiguous()  # (L=h*w, d_model)
        self.register_buffer('pos_emb_baked', pos)
        # Bake grid_hws as a constant buffer.  Rope2DReal uses .item() on grid_hws and
        # sdpa_packed ignores cu_seqlens, so the exporter would otherwise constant-fold
        # the grid_hws input out of the graph and ORT/TRT would reject it as a dead input.
        self.register_buffer('grid_hws_baked',
                             torch.tensor([[int(grid_h), int(grid_w)]], dtype=torch.int32))
        self.merge_kh, self.merge_kw = vit.merge_kernel_size
    def forward(self, pixel_values):
        x = self.patch_proj(pixel_values).view(pixel_values.size(0), -1)  # (L, d_model)
        x = x + self.pos_emb_baked
        x = self.encoder(x, self.grid_hws_baked)
        return x   # (L, d_model)  pre-merger

# Build wrapper using the actual demo resolution.
_grid_h, _grid_w = int(inputs['image_grid_hws'][0,0]), int(inputs['image_grid_hws'][0,1])
# Constants reused by downstream cells (LLM prefill driver, TRT profiles, orchestrator).
L_pre_fixed  = _grid_h * _grid_w                 # patches per image (= 1656 for 36x46)
L_post_fixed = L_pre_fixed // (2 * 2)            # post-merger image tokens (= 414)
print(f'baking pos_emb for grid_hws=({_grid_h},{_grid_w})  L_pre={L_pre_fixed}  L_post={L_post_fixed}')
vit_wrap = VisionForExport(model.vision_model, _grid_h, _grid_w).eval().to('cuda')

px = inputs['pixel_values'].detach()
gh = inputs['image_grid_hws'].detach().to(torch.int32)
with torch.inference_mode():
    vit_out_pre = vit_wrap(px)
print('vision input ', px.shape, px.dtype, '  grid', gh.tolist())
print('vision output (pre-merger)', vit_out_pre.shape, vit_out_pre.dtype)

VIT_PRE_DIM = vit_out_pre.shape[-1]   # 1152 (== hidden_size before patch_merger 4x cat)
VIT_FEAT_DIM = VIT_PRE_DIM * vit_wrap.merge_kh * vit_wrap.merge_kw  # 4608 (projector input dim, post Python merger)

def python_patch_merger(x_np, gh_np, kh=2, kw=2):
    """Numerically identical to canonical patch_merger for a single image.
    Canonical:
        seq.view(nh, kh, nw, kw, d).permute(0,2,1,3,4).contiguous().view(nh*nw, -1)
    where seq has shape (h*w, d) with row-major (outer_h, outer_w) layout.
    Go directly from (L, d) to (nh, kh, nw, kw, d) — the intermediate (h, w, d)
    reshape is unnecessary and would be a silent permutation if the input were
    non-contiguous. np.ascontiguousarray guards against the latter.
    """
    h, w = int(gh_np[0, 0]), int(gh_np[0, 1])
    nh, nw = h // kh, w // kw
    d = x_np.shape[-1]
    x = np.ascontiguousarray(x_np)
    return (x.reshape(nh, kh, nw, kw, d)
             .transpose(0, 2, 1, 3, 4)
             .reshape(nh * nw, kh * kw * d))

VIT_ONNX = ONNX_DIR/'vision.onnx'
if VIT_ONNX.exists() and VIT_ONNX.stat().st_size>0:
    print('cached:', VIT_ONNX)
else:
    torch.onnx.export(
        vit_wrap, (px,), str(VIT_ONNX),
        input_names=['pixel_values'],
        output_names=['vit_feats'],   # pre-merger; orchestrator runs python_patch_merger
        dynamic_axes={'pixel_values':{0:'L_pre'}, 'vit_feats':{0:'L_pre'}},
        opset_version=17, do_constant_folding=True, dynamo=False,
    )
    print('saved', VIT_ONNX, '·', VIT_ONNX.stat().st_size/1e6, 'MB')


## 6 · Export sub-graph 2: MLP projector (mlp1)

In [ ]:
class ProjectorForExport(torch.nn.Module):
    def __init__(self, mlp1): super().__init__(); self.mlp1 = mlp1
    def forward(self, x): return self.mlp1(x)

proj_wrap = ProjectorForExport(model.mlp1).eval().to('cuda')
PROJ_DIM_IN = VIT_FEAT_DIM
dummy_proj = torch.zeros(32, PROJ_DIM_IN, dtype=REF_DTYPE, device='cuda')
with torch.inference_mode():
    dummy_out = proj_wrap(dummy_proj)
PROJ_DIM_OUT = dummy_out.shape[-1]
print(f'projector  in={PROJ_DIM_IN}  out={PROJ_DIM_OUT}')

PROJ_ONNX = ONNX_DIR/'projector.onnx'
if PROJ_ONNX.exists() and PROJ_ONNX.stat().st_size>0:
    print('cached:', PROJ_ONNX)
else:
    torch.onnx.export(
        proj_wrap, (dummy_proj,), str(PROJ_ONNX),
        input_names=['vit_feats_4x'], output_names=['proj_feats'],
        dynamic_axes={'vit_feats_4x':{0:'L_post'},'proj_feats':{0:'L_post'}},
        opset_version=17, do_constant_folding=True, dynamo=False,
    )
    print('saved', PROJ_ONNX, '·', PROJ_ONNX.stat().st_size/1e6, 'MB')

## 7 · Export sub-graphs 3 & 4: LLM prefill + dynamic-K decode (KV cache externalised)

The decode graph accepts `K ∈ {1, 6}` so a single engine serves both AR steps and the PBD multi-token-prediction block. Each layer contributes one `past_k_i` + `past_v_i` input and one `present_k_i` + `present_v_i` output.

Because the LLM is ~6 GB FP16, we use `onnx.save_model(save_as_external_data=True)` to break it across side-car weight files (the 2 GB protobuf limit applies to the graph proto, not the weights).

In [ ]:
tc = config.text_config
H   = tc.hidden_size
L_N = tc.num_hidden_layers
Hkv = tc.num_key_value_heads
Dh  = H // tc.num_attention_heads
V_  = tc.vocab_size
print(f'Qwen2.5 LM: layers={L_N} hidden={H} kv_heads={Hkv} head_dim={Dh} vocab={V_}')

lm_main = model.language_model.model if hasattr(model.language_model, 'model') else model.language_model
lm_head = model.language_model.lm_head
embed_tokens = lm_main.embed_tokens

class LLMPrefill(torch.nn.Module):
    """Wraps Qwen2Model.forward for prefill.
    Canonical contract (modeling_qwen2.py:1184-1227):
      - `input_ids` XOR `inputs_embeds` — never both (line 1200 raises ValueError)
      - Vision tokens are scattered into embeddings INSIDE the model via
        `image_processing(input_ids, visual_features, image_token_index)` (line 1167)
      - The block-mask path at line 1253 reads `input_ids[0][-1].item()` to decide
        MTP vs AR mode — for prefill we want the early-return branch, which fires
        when the last input_id is NOT the text_mask_token_id (i.e., normal prompts
        ending in real text — the case the chat template produces).
    """
    def __init__(self, m, h, image_token_index):
        super().__init__()
        self.m = m
        self.h = h
        self.image_token_index = int(image_token_index)
    def forward(self, input_ids, visual_features, position_ids, attention_mask):
        o = self.m(
            input_ids=input_ids,
            visual_features=visual_features,
            image_token_index=self.image_token_index,
            position_ids=position_ids,
            attention_mask=attention_mask,
            use_cache=True,
            return_dict=True,
        )
        # Match canonical Qwen2ForCausalLM.forward (line 1471): unconditional fp32 cast.
        logits = self.h(o.last_hidden_state).float()
        flat=[]
        for (k,v) in o.past_key_values:
            flat.append(k); flat.append(v)
        return (logits, *flat)

class LLMDecode(torch.nn.Module):
    """Wraps Qwen2Model.forward for decode (AR K=1 or MTP K=6). No visual_features.
    input_ids is required because _prepare_block_mask_for_inference (line 1253)
    reads input_ids[0][-1] when seq_length > 1 to decide between AR early-return
    and SDLM block-mask construction. For MTP traces we put text_mask_token_id at
    the last position so the block-mask branch is captured by the export.
    """
    def __init__(self, m, h, n):
        super().__init__()
        self.m = m
        self.h = h
        self.n = n
    def forward(self, input_ids, position_ids, attention_mask, *past_kv):
        pkv = [(past_kv[2*i], past_kv[2*i+1]) for i in range(self.n)]
        o = self.m(
            input_ids=input_ids,
            position_ids=position_ids,
            attention_mask=attention_mask,
            past_key_values=tuple(pkv),
            use_cache=True,
            return_dict=True,
        )
        logits = self.h(o.last_hidden_state).float()
        flat=[]
        for (k,v) in o.past_key_values:
            flat.append(k); flat.append(v)
        return (logits, *flat)

def export_with_external_data(wrap, args, onnx_path, **kw):
    """Export to a temp dir, then re-save with external data so each graph fits the protobuf cap."""
    onnx_path = Path(onnx_path)
    if onnx_path.exists() and onnx_path.stat().st_size>0:
        print('cached:', onnx_path); return
    tmp = onnx_path.parent / (onnx_path.stem + '_tmp')
    tmp.mkdir(exist_ok=True)
    tmp_file = tmp/'model.onnx'
    # Force the legacy TorchScript-based exporter. The modern dynamo path raises
    # GuardOnDataDependentSymNode on any `.item()` / `int(tensor_scalar)` call
    # (we have those in Rope2DReal.get_freqs_cis); the legacy path bakes the
    # specialised int constant at trace time, which is exactly what we want
    # since VisionForExport is already fixed-resolution.
    kw.setdefault('dynamo', False)
    torch.onnx.export(wrap, args, str(tmp_file), **kw)
    m = onnx.load(str(tmp_file), load_external_data=True)
    onnx.save_model(m, str(onnx_path),
                    save_as_external_data=True,
                    all_tensors_to_one_file=True,
                    location=onnx_path.stem + '.bin')
    shutil.rmtree(tmp, ignore_errors=True)
    print('saved', onnx_path, '·', onnx_path.stat().st_size/1e6, 'MB (+ side-car)')

In [ ]:
# ---- Prefill: dynamic S ----
PREFILL_ONNX = ONNX_DIR/'llm_prefill.onnx'
n_img_trace = L_post_fixed                       # 414 for the demo grid (36x46/4)
TEXT_PAD_TRACE = 64                              # representative text tokens around the image block
S_trace = n_img_trace + TEXT_PAD_TRACE * 2       # = 542 for demo — must fit the slice below
# Build dummy input_ids: image_token_index at n_img_trace contiguous positions, rest 0.
# CRITICAL: ids_d[0, -1] must NOT be text_mask_token_id so the block-mask early-return
# branch is the one captured by the trace (matches the chat-template behaviour at runtime).
ids_d = torch.zeros(1, S_trace, dtype=torch.long, device='cuda')
ids_d[0, TEXT_PAD_TRACE:TEXT_PAD_TRACE+n_img_trace] = int(config.image_token_index)
# Sanity-check: exactly n_img_trace image-token positions, last position is non-mask.
_n_img_check = int((ids_d == int(config.image_token_index)).sum().item())
assert _n_img_check == n_img_trace, f'image-token count mismatch: {_n_img_check} vs {n_img_trace}'
assert int(ids_d[0, -1].item()) != int(config.text_config.text_mask_token_id), 'last token would trigger SDLM block-mask path'
# Visual features: (n_img_trace, hidden_size). Engine handles variable counts via dynamic axis.
vf_d  = torch.zeros(n_img_trace, H, dtype=REF_DTYPE, device='cuda')
pos_d = torch.arange(S_trace, dtype=torch.long, device='cuda')[None]
msk_d = torch.ones(1, S_trace, dtype=torch.long, device='cuda')
kv_out_names = sum([[f'present_k_{i}', f'present_v_{i}'] for i in range(L_N)], [])
dyn_pre = {
    'input_ids':       {1: 'S'},
    'visual_features': {0: 'L_post'},
    'position_ids':    {1: 'S'},
    'attention_mask':  {1: 'S'},
    'logits':          {1: 'S'},
    **{n:{2:'S'} for n in kv_out_names},
}
export_with_external_data(
    LLMPrefill(lm_main, lm_head, int(config.image_token_index)).eval(),
    (ids_d, vf_d, pos_d, msk_d),
    PREFILL_ONNX,
    input_names=['input_ids','visual_features','position_ids','attention_mask'],
    output_names=['logits', *kv_out_names],
    dynamic_axes=dyn_pre,
    opset_version=17, do_constant_folding=False,   # save peak RAM
)


In [ ]:
# ---- Decode: dynamic K, dynamic past_seq ----
DECODE_ONNX = ONNX_DIR/'llm_decode.onnx'
K_trace, P_trace = 6, 32
# input_ids: last position == text_mask_token_id so the SDLM block-mask branch is
# traced for the K_trace=6 MTP window. (For runtime K=1 the canonical short-circuits
# on seq_length==1 regardless of input_ids[-1] value.)
ids_dec = torch.full((1, K_trace), int(config.text_config.text_mask_token_id), dtype=torch.long, device='cuda')
ids_dec[0, 0] = 0   # non-mask first position (represents the last committed real token)
pd_ = torch.arange(P_trace, P_trace+K_trace, dtype=torch.long, device='cuda')[None]
md  = torch.ones(1, P_trace+K_trace, dtype=torch.long, device='cuda')
past_args = []
for _ in range(L_N):
    past_args.append(torch.zeros(1, Hkv, P_trace, Dh, dtype=REF_DTYPE, device='cuda'))
    past_args.append(torch.zeros(1, Hkv, P_trace, Dh, dtype=REF_DTYPE, device='cuda'))
in_kv_names  = sum([[f'past_k_{i}',    f'past_v_{i}']    for i in range(L_N)], [])
out_kv_names = sum([[f'present_k_{i}', f'present_v_{i}'] for i in range(L_N)], [])
dyn_dec = {
    'input_ids':       {1: 'K'},
    'position_ids':    {1: 'K'},
    'attention_mask':  {1: 'P_plus_K'},
    'logits':          {1: 'K'},
    **{n:{2:'P'}        for n in in_kv_names},
    **{n:{2:'P_plus_K'} for n in out_kv_names},
}
export_with_external_data(
    LLMDecode(lm_main, lm_head, L_N).eval(),
    (ids_dec, pd_, md, *past_args),
    DECODE_ONNX,
    input_names=['input_ids','position_ids','attention_mask', *in_kv_names],
    output_names=['logits', *out_kv_names],
    dynamic_axes=dyn_dec,
    opset_version=17, do_constant_folding=False,
)


## 7b · INT4 AWQ quantisation — single-file ONNX (optional)

Re-export the LLM with NVIDIA Model Optimizer's INT4 AWQ.  3 B params × 4 bits packed = ~1.5 GB of weights, which fits **inside** the 2 GB Protobuf cap — so each of `llm_prefill_int4.onnx` and `llm_decode_int4.onnx` becomes a single literal file with **no side-car `.bin`**.

Quality cost on grounding is near-zero because the box-coord output is sampled from a discrete vocab (151677–152677) rather than regressed.  Throughput improves ~1.8–2.0× on H100 / L40S.

Toggle this whole section off by setting `USE_INT4_LLM = False` in the cell below.

In [ ]:
USE_INT4_LLM = True   # set False to skip the quantisation pass and stay on FP16
PREFILL_INT4_ONNX = ONNX_DIR/'llm_prefill_int4.onnx'
DECODE_INT4_ONNX  = ONNX_DIR/'llm_decode_int4.onnx'

if USE_INT4_LLM and not (PREFILL_INT4_ONNX.exists() and DECODE_INT4_ONNX.exists()):
    if not have('modelopt'):
        sh('pip -q install "nvidia-modelopt[torch]" --extra-index-url https://pypi.nvidia.com', check=False)
        importlib.invalidate_caches()
    import modelopt.torch.quantization as mtq
    print('modelopt version:', importlib.import_module('modelopt').__version__)

    # ---- Calibration loop: 5 representative VLM forward passes ----
    # AWQ-lite calibrates per-channel activation amax for every Linear's input.
    # AWQ-lite calls forward_loop(model) MANY times internally (cache-stats pass +
    # per-projection alpha-search), so the loop must be:
    #   (a) safe to re-invoke and side-effect-free w.r.t. its captured state,
    #   (b) diverse and long enough that every Linear sees real activations,
    #   (c) NOT wrapped in torch.inference_mode() — that disables view-tracking
    #       hooks modelopt installs. Use torch.no_grad() instead.
    # Vision features are NOT needed for LM-weight AWQ: the LM's Linear inputs are
    # post-LayerNorm features whose magnitude is dominated by the LN scale, not by
    # whether the upstream embedding came from a text token or a scattered visual
    # token. Text-only calibration with a varied corpus is the canonical approach.
    CALIB_TEXTS = [
        'Detect all cats. Return bounding boxes.',
        'Find every person in the image and bound each one.',
        'Where are the red cars? Provide bounding boxes for each.',
        'List bounding boxes for every animal visible.',
        'Locate the chair nearest the window.',
        'Identify the traffic signs and bound them.',
        'Find any text in the image; bound each text region.',
        'Point to the largest object in the scene.',
        'Detect every pedestrian, cyclist, and vehicle.',
        'Bound every face you can see.',
        # Pure-text passages widen the activation distribution:
        'The quick brown fox jumps over the lazy dog near the riverbank.',
        'In a hole in the ground there lived a hobbit. Not a nasty, dirty, wet hole.',
        'It was the best of times, it was the worst of times, it was the age of wisdom.',
        'To be, or not to be, that is the question: whether tis nobler in the mind.',
        'All happy families are alike; each unhappy family is unhappy in its own way.',
        # Coordinate-token vocabulary touches: helps the LM see the box/coord token range.
        '<box><x1><y1><x2><y2></box> represents a bounding-box coordinate quadruple.',
        '<ref>object name</ref> tokens precede each <box>...</box> block in the output.',
    ]
    def calibration_loop(m=None):
        # Note: modelopt may call this 5-100+ times; keep it side-effect-free.
        for text in CALIB_TEXTS:
            ids = tokenizer(text, return_tensors='pt').input_ids.to('cuda')
            with torch.no_grad():
                model.language_model.model(
                    input_ids=ids,
                    position_ids=torch.arange(ids.shape[1], device=ids.device)[None, :],
                    attention_mask=torch.ones_like(ids),
                    use_cache=False, return_dict=True,
                )

    print('Running INT4 AWQ calibration on the language model (~1-3 min) ...')
    t0 = time.time()
    mtq.quantize(model.language_model, mtq.INT4_AWQ_CFG, forward_loop=calibration_loop)
    print(f'  quantised in {time.time()-t0:.1f}s')

    # ---- Re-export prefill ----
    print('Exporting INT4 prefill ...')
    torch.onnx.export(
        LLMPrefill(lm_main, lm_head, int(config.image_token_index)).eval(),
        (ids_d, vf_d, pos_d, msk_d),
        str(PREFILL_INT4_ONNX),
        input_names=['input_ids','visual_features','position_ids','attention_mask'],
        output_names=['logits', *kv_out_names],
        dynamic_axes=dyn_pre,
        opset_version=17, do_constant_folding=False, dynamo=False,
    )
    sz = PREFILL_INT4_ONNX.stat().st_size/1e9
    print(f'  llm_prefill_int4.onnx  =  {sz:.2f} GB  ({"single-file ✓" if sz < 2.0 else "still needs side-car"})')

    # If a side-car snuck in (e.g. kv-cache scratch is still FP16 and pushes us >2GB), bundle to one .bin so we still ship 2 files max.
    sidecar = list(PREFILL_INT4_ONNX.parent.glob(PREFILL_INT4_ONNX.stem + '*.data'))
    if sidecar:
        print('  bundling side-car ->', sidecar)
        m = onnx.load(str(PREFILL_INT4_ONNX), load_external_data=True)
        onnx.save_model(m, str(PREFILL_INT4_ONNX),
                        save_as_external_data=True, all_tensors_to_one_file=True,
                        location=PREFILL_INT4_ONNX.stem + '.bin')

    # ---- Re-export decode ----
    print('Exporting INT4 decode ...')
    torch.onnx.export(
        LLMDecode(lm_main, lm_head, L_N).eval(),
        (ids_dec, pd_, md, *past_args),
        str(DECODE_INT4_ONNX),
        input_names=['input_ids','position_ids','attention_mask', *in_kv_names],
        output_names=['logits', *out_kv_names],
        dynamic_axes=dyn_dec,
        opset_version=17, do_constant_folding=False, dynamo=False,
    )
    sz = DECODE_INT4_ONNX.stat().st_size/1e9
    print(f'  llm_decode_int4.onnx   =  {sz:.2f} GB  ({"single-file ✓" if sz < 2.0 else "still needs side-car"})')
    sidecar = list(DECODE_INT4_ONNX.parent.glob(DECODE_INT4_ONNX.stem + '*.data'))
    if sidecar:
        m = onnx.load(str(DECODE_INT4_ONNX), load_external_data=True)
        onnx.save_model(m, str(DECODE_INT4_ONNX),
                        save_as_external_data=True, all_tensors_to_one_file=True,
                        location=DECODE_INT4_ONNX.stem + '.bin')
elif USE_INT4_LLM:
    print('INT4 ONNX files already exist — re-using cached:', PREFILL_INT4_ONNX, DECODE_INT4_ONNX)
else:
    print('INT4 disabled (USE_INT4_LLM=False); TRT build will use FP16 ONNX.')

## 8 · ONNX-runtime parity on vision + projector

We don't host the 6 GB LLM under ORT (it would double VRAM). Instead we validate the two small graphs end-to-end and rely on TRT for the LLM.

In [ ]:
providers = [('CUDAExecutionProvider', {'device_id':0}), 'CPUExecutionProvider']
vit_sess  = ort.InferenceSession(str(VIT_ONNX),  providers=providers)
proj_sess = ort.InferenceSession(str(PROJ_ONNX), providers=providers)

px_np = px.detach().cpu().numpy().astype(np.float16)
gh_np = gh.detach().cpu().numpy().astype(np.int32)

# ---- Vision ----
vit_ort_pre = vit_sess.run(None, {'pixel_values': px_np})[0]   # (L, 1152) pre-merger (grid_hws baked into engine)
with torch.inference_mode():
    vit_pt_pre = vit_wrap(px).cpu().numpy()                                        # pre-merger PT
def _err_stats(a, b, name):
    """Distribution-aware FP16 parity report. max|err| alone is meaningless across ~2M
    elements where ORT and PyTorch can each pick different SDPA kernel paths; report
    mean/median/p99/max and use a realistic threshold on the central tendency."""
    d = np.abs(a.astype(np.float32) - b.astype(np.float32)).reshape(-1)
    mean, med, p99, mx = float(d.mean()), float(np.median(d)), float(np.percentile(d, 99)), float(d.max())
    a_abs = np.abs(a.astype(np.float32)).mean() + 1e-6
    print(f'{name:>32s}  mean={mean:.3e}  median={med:.3e}  p99={p99:.3e}  max={mx:.3e}  '
          f'(rel mean={mean/a_abs:.2%})')
    return mean, med, p99, mx, a_abs

err_v_stats = _err_stats(vit_ort_pre, vit_pt_pre, 'vision (pre-merger)')

# Numerical patch_merger (deterministic reshape+permute; identical PT vs np).
vit_ort_4x = python_patch_merger(vit_ort_pre, gh_np, vit_wrap.merge_kh, vit_wrap.merge_kw)

# Cross-check against canonical post-merger forward (which returns List[Tensor]).
with torch.inference_mode():
    pt_post_list = model.vision_model(pixel_values=px, grid_hws=gh)
    vit_pt_4x = torch.cat(pt_post_list, dim=0).cpu().numpy()
err_merge_stats = _err_stats(vit_ort_4x, vit_pt_4x, 'vision (post-merger)')

# ---- Projector ----
proj_ort = proj_sess.run(None, {'vit_feats_4x': vit_ort_4x.astype(np.float16)})[0]
with torch.inference_mode():
    proj_pt = model.mlp1(torch.from_numpy(vit_ort_4x).cuda().to(REF_DTYPE)).cpu().numpy()
err_p_stats = _err_stats(proj_ort, proj_pt, 'projector')

# Distribution-aware parity check. FP16 tolerances after 27 transformer layers:
#   mean error <  5e-2  (central tendency must be tight)
#   p99 error  <  1.0   (worst few percent can drift due to SDPA kernel diff)
#   max error  reported but NOT asserted on — a single outlier in ~2M values is normal
mean_v = err_v_stats[0];  p99_v = err_v_stats[2]
mean_p = err_p_stats[0];  p99_p = err_p_stats[2]
ok_v = (mean_v < 5e-2) and (p99_v < 1.0)
ok_p = (mean_p < 5e-2) and (p99_p < 1.0)
if ok_v and ok_p:
    print('PARITY OK  (mean<5e-2, p99<1.0; FP16-realistic tolerance)')
else:
    print('PARITY WARNING — central-tendency drift exceeds threshold:')
    if not ok_v: print(f'  vision: mean={mean_v:.3e} p99={p99_v:.3e}')
    if not ok_p: print(f'  projector: mean={mean_p:.3e} p99={p99_p:.3e}')
    print('  This often still produces correct boxes end-to-end; the real test is cell 36 (TRT vs PT box IoU).')


## 9 · TensorRT engine builds (FP16, optimisation profiles)

In [ ]:
def build_engine(onnx_path, profile_spec, fp16=True, workspace_gb=4, name='engine'):
    """profile_spec: {input_name: (min, opt, max)}"""
    out = TRT_DIR/f'{name}.engine'
    if out.exists():
        print('cached:', out); return out
    builder = trt.Builder(TRT_LOGGER)
    net = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(net, TRT_LOGGER)
    # parse_from_file (not parse(bytes)) so the parser can locate side-car external-data
    # files (.bin) that sit next to the .onnx — required for FP16 LLM exports and INT4
    # exports that use onnx.save_model(save_as_external_data=True).
    if not parser.parse_from_file(str(onnx_path)):
        for i in range(parser.num_errors): print(parser.get_error(i))
        raise RuntimeError(f'ONNX parse failed for {onnx_path}')
    cfg = builder.create_builder_config()
    cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, workspace_gb << 30)
    if fp16: cfg.set_flag(trt.BuilderFlag.FP16)
    prof = builder.create_optimization_profile()
    for n,(lo,opt,hi) in profile_spec.items():
        prof.set_shape(n, lo, opt, hi)
    cfg.add_optimization_profile(prof)
    print(f'Building {name} (fp16={fp16}) ...')
    t0 = time.time()
    plan = builder.build_serialized_network(net, cfg)
    if plan is None: raise RuntimeError(f'TRT build returned None for {name}')
    out.write_bytes(plan)
    print(f'  built in {time.time()-t0:.1f}s -> {out} ({out.stat().st_size/1e9:.2f} GB)')
    return out

# Vision profile.  The baked pos_emb in VisionForExport fixes L_pre to the
# resolution we exported at — we still expose L_pre as a dynamic axis but
# point min=opt=max at that one value.  Re-export at a different resolution
# by re-running the vision export cell with new (grid_h, grid_w).
# L_pre_fixed already defined in the vision-export cell.
if px.dim()==4:
    VIT_PROF = {
        'pixel_values': ((L_pre_fixed,3,14,14),(L_pre_fixed,3,14,14),(L_pre_fixed,3,14,14)),
    }
else:
    _shape_tail = tuple(px.shape[1:])
    VIT_PROF = {
        'pixel_values': ((L_pre_fixed,)+_shape_tail,(L_pre_fixed,)+_shape_tail,(L_pre_fixed,)+_shape_tail),
    }
# L_post_fixed already defined in the vision-export cell.
VIT_ENG = build_engine(str(VIT_ONNX), VIT_PROF, workspace_gb=2, name='vision')

PROJ_PROF = {'vit_feats_4x':((L_post_fixed, PROJ_DIM_IN),(L_post_fixed, PROJ_DIM_IN),(L_post_fixed, PROJ_DIM_IN))}
PROJ_ENG = build_engine(str(PROJ_ONNX), PROJ_PROF, workspace_gb=1, name='projector')

In [ ]:
# LLM engines: prefer INT4 if quantisation cell ran, otherwise FP16.
# Note: modelopt's INT4_AWQ_CFG + torch.onnx.export produces ONNX that stores INT4 weights
# as packed-INT8 with block-wise DequantizeLinear scales. TensorRT 10.7's parser rejects
# this layout (it requires `input.getType() == DataType::kINT4` for blocked scales).
# Workarounds: (a) TRT-LLM (`trtllm-build --quant-config awq_int4`), (b) modelopt.onnx
# .quantization.quantize_static on the FP16 ONNX, (c) post-process to repack as INT4 dtype.
# For now: try INT4, fall back to FP16 on parse failure (FP16 build still produces a working engine).
PREFILL_ENG = DECODE_ENG = None
_have_int4 = (PREFILL_INT4_ONNX.exists() and DECODE_INT4_ONNX.exists()) if 'PREFILL_INT4_ONNX' in globals() else False

# Profile names MUST match the ONNX graph's input names (cell 20).  Mismatched
# names cause TRT's build_serialized_network to return None *silently* — no
# log entry at any verbosity level.  The wrappers were refactored to the
# canonical input_ids+visual_features contract; the profile must follow.
PREFILL_PROF = {
    'input_ids':       ((1,16),       (1,1024),       (1,4096)),
    'visual_features': ((L_post_fixed,H),(L_post_fixed,H),(L_post_fixed,H)),  # fixed by baked pos_emb resolution
    'position_ids':    ((1,16),       (1,1024),       (1,4096)),
    'attention_mask':  ((1,16),       (1,1024),       (1,4096)),
}
DECODE_PROF = {
    'input_ids':      ((1,1),(1,6),  (1,6)),       # K∈{1 AR, 6 MTP}
    'position_ids':   ((1,1),(1,6),  (1,6)),
    'attention_mask': ((1,1),(1,1030),(1,4102)),   # P+K range
}
for i in range(L_N):
    DECODE_PROF[f'past_k_{i}'] = ((1,Hkv,0,Dh),(1,Hkv,1024,Dh),(1,Hkv,4096,Dh))
    DECODE_PROF[f'past_v_{i}'] = ((1,Hkv,0,Dh),(1,Hkv,1024,Dh),(1,Hkv,4096,Dh))

def _free_pt_model_for_build():
    """Free the PyTorch model from VRAM so TRT can use the full GPU for its build
    workspace. The 3B FP16 model is ~6 GB; TRT's build needs ~12-18 GB of
    workspace to optimise the LLM prefill graph. On a 24 GB L4 the two together
    overflow. We trade the PT fallback path (it only fires if the TRT build
    itself fails, so once the build succeeds we never need it)."""
    global model
    import gc as _gc_local
    freed = False
    if 'model' in globals():
        try:
            model.cpu()  # move weights off GPU first so cuda cache can reclaim
        except Exception: pass
        try:
            del model
            freed = True
        except Exception: pass
    _gc_local.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    if freed:
        free_gb, total_gb = torch.cuda.mem_get_info()
        print(f'  freed PT model. GPU free: {free_gb/1e9:.1f} / {total_gb/1e9:.1f} GB')
    return freed

def _try_build(prefill_src, decode_src, tag, ws):
    """Build prefill + decode engines. On TRT 'build returned None' (typically OOM
    when the PT model is co-resident), free the PT model and retry with a slightly
    smaller workspace pool."""
    def _one(src, name, w):
        prof = PREFILL_PROF if 'prefill' in name else DECODE_PROF
        return build_engine(str(src), prof, workspace_gb=w, name=name)
    try:
        pre = _one(prefill_src, f'llm_prefill_{tag}', ws)
    except RuntimeError as e:
        if 'build returned None' not in str(e): raise
        print(f'  prefill build returned None — freeing PT model and retrying with ws={ws-2} ...')
        _free_pt_model_for_build()
        pre = _one(prefill_src, f'llm_prefill_{tag}', max(2, ws-2))
    try:
        dec = _one(decode_src, f'llm_decode_{tag}', ws)
    except RuntimeError as e:
        if 'build returned None' not in str(e): raise
        print(f'  decode build returned None — freeing PT model and retrying with ws={ws-2} ...')
        _free_pt_model_for_build()
        dec = _one(decode_src, f'llm_decode_{tag}', max(2, ws-2))
    return pre, dec

LLM_INT4 = False
if _have_int4:
    print(f'LLM source: int4  ({PREFILL_INT4_ONNX.name}, {DECODE_INT4_ONNX.name})')
    try:
        PREFILL_ENG, DECODE_ENG = _try_build(PREFILL_INT4_ONNX, DECODE_INT4_ONNX, 'int4', ws=4)
        LLM_INT4 = True
    except RuntimeError as e:
        msg = str(e).lower()
        if 'parse' in msg or 'dequantize' in msg or 'block' in msg:
            print('  INT4 parse failed (modelopt ONNX uses INT8-packed INT4 layout, TRT expects native INT4 dtype).')
            print('  Falling back to FP16 LLM engine. To get true INT4, use TRT-LLM or modelopt.onnx.quantization.quantize_static.')
            PREFILL_ENG = DECODE_ENG = None
        else:
            raise

if PREFILL_ENG is None and ENABLE_LLM_TRT:
    print(f'LLM source: fp16  ({PREFILL_ONNX.name}, {DECODE_ONNX.name})')
    PREFILL_ENG, DECODE_ENG = _try_build(PREFILL_ONNX, DECODE_ONNX, 'fp16', ws=8)
elif PREFILL_ENG is None:
    print(f'Skipping LLM TRT build (VRAM={VRAM_GB:.1f} GB < 22 GB and no INT4 path that builds cleanly).')
    print('PyTorch LM will be used in the orchestrator fallback path.')


## 10 · NVIDIA-architect deep-dive: custom plugin for packed-varlen attention

The SDPA fallback we used in §3 is correct but pays an `O(L²)` mask cost on >2.5 K-resolution inputs; the original `flash_attn_varlen_func` avoids that with a fused CUDA kernel. We provide a TRT plugin source skeleton that wraps the same kernel — drop the compiled `.so` into `LD_PRELOAD` to upgrade the vision engine without re-exporting ONNX.

In [ ]:
PLUGIN_SRC = r'''// packed_varlen_attn_plugin.cu — TensorRT 10 plugin for flash-attn varlen.
//  build:  nvcc -shared -Xcompiler -fPIC -I$TRT_INC -I$CUDA_HOME/include \
//          -L$FLASH_ATTN_LIB -lflash_attn packed_varlen_attn_plugin.cu \
//          -o packed_varlen_attn_plugin.so
#include "NvInferRuntimePlugin.h"
#include <cuda_runtime.h>

extern "C" void flash_attn_varlen_fwd(
    const void* q, const void* k, const void* v,
    const int32_t* cu_q, const int32_t* cu_k,
    int max_seqlen_q, int max_seqlen_k,
    int total_q, int total_k, int num_heads, int head_dim,
    void* out, cudaStream_t stream);

using namespace nvinfer1;
class PackedVarlenAttn : public IPluginV3, public IPluginV3OneCore,
                         public IPluginV3OneBuild, public IPluginV3OneRuntime {
public:
    const char* getPluginName()      const noexcept { return "PackedVarlenAttn"; }
    const char* getPluginVersion()   const noexcept { return "1"; }
    const char* getPluginNamespace() const noexcept { return ""; }
    int  enqueue(const PluginTensorDesc* in, const PluginTensorDesc*,
                 const void* const* ins, void* const* outs,
                 void*, cudaStream_t stream) noexcept {
        const auto& Q = in[0]; const auto& KK = in[1]; const auto& V = in[2];
        flash_attn_varlen_fwd(ins[0], ins[1], ins[2],
            reinterpret_cast<const int32_t*>(ins[3]),
            reinterpret_cast<const int32_t*>(ins[4]),
            Q.dims.d[0], KK.dims.d[0], Q.dims.d[0], KK.dims.d[0],
            Q.dims.d[1], Q.dims.d[2], outs[0], stream);
        return 0;
    }
    /* … getOutputShapes(), supportsFormatCombination(), clone(), serialize() … */
};
class Creator : public IPluginCreatorV3One {
    const char* getPluginName()    const noexcept { return "PackedVarlenAttn"; }
    const char* getPluginVersion() const noexcept { return "1"; }
    IPluginV3* createPlugin(const char*, const PluginFieldCollection*,
                             TensorRTPhase) noexcept { return new PackedVarlenAttn(); }
};
REGISTER_TENSORRT_PLUGIN(Creator);
'''
(WORK/'packed_varlen_attn_plugin.cu').write_text(PLUGIN_SRC)
print('Plugin source ->', WORK/'packed_varlen_attn_plugin.cu')
print('To enable in production:')
print('  1) compile the .so against your flash-attn build')
print('  2) ctypes.CDLL("./packed_varlen_attn_plugin.so") before deserializing the engine')
print('  3) re-export modeling_vit.multihead_attention as a torch op so ONNX emits a PackedVarlenAttn node')

## 11 · TRT orchestrator (replaces the canonical generate())

In [ ]:
from cuda.bindings import runtime as cudart

def _ck(ret):
    err = ret[0]
    if err != cudart.cudaError_t.cudaSuccess:
        raise RuntimeError(f'CUDA error: {cudart.cudaGetErrorString(err)[1]}')
    return ret[1] if len(ret)>1 else None

class TRTEngine:
    def __init__(self, path):
        rt = trt.Runtime(TRT_LOGGER)
        self.engine = rt.deserialize_cuda_engine(Path(path).read_bytes())
        self.ctx = self.engine.create_execution_context()
        self.io = [self.engine.get_tensor_name(i) for i in range(self.engine.num_io_tensors)]
        self.is_in = {n: self.engine.get_tensor_mode(n)==trt.TensorIOMode.INPUT for n in self.io}
        self.dtype = {n: trt.nptype(self.engine.get_tensor_dtype(n)) for n in self.io}
    def __call__(self, feed):
        bufs = []; outs = {}
        for n,arr in feed.items():
            arr = np.ascontiguousarray(arr.astype(self.dtype[n]))
            dptr = _ck(cudart.cudaMalloc(arr.nbytes))
            _ck(cudart.cudaMemcpy(dptr, arr.ctypes.data, arr.nbytes,
                                  cudart.cudaMemcpyKind.cudaMemcpyHostToDevice))
            self.ctx.set_tensor_address(n, int(dptr))
            ok = self.ctx.set_input_shape(n, arr.shape)
            if not ok:
                # TRT returns False (not an exception) when the requested shape is
                # outside the engine's optimisation profile.  Silently running the
                # engine in that state produces stale output — surface it.
                raise RuntimeError(f'set_input_shape({n}, {arr.shape}) rejected — outside profile range')
            bufs.append(dptr)
        for n in self.io:
            if self.is_in[n]: continue
            shape = tuple(self.ctx.get_tensor_shape(n))
            arr = np.empty(shape, dtype=self.dtype[n])
            dptr = _ck(cudart.cudaMalloc(arr.nbytes))
            self.ctx.set_tensor_address(n, int(dptr))
            outs[n] = (dptr, arr); bufs.append(dptr)
        stream = _ck(cudart.cudaStreamCreate())
        self.ctx.execute_async_v3(int(stream))
        _ck(cudart.cudaStreamSynchronize(stream))
        result = {}
        for n,(dptr,arr) in outs.items():
            _ck(cudart.cudaMemcpy(arr.ctypes.data, dptr, arr.nbytes,
                                  cudart.cudaMemcpyKind.cudaMemcpyDeviceToHost))
            result[n] = arr
        for d in bufs:
            cudart.cudaFree(d)
        cudart.cudaStreamDestroy(stream)
        return result

vit_e   = TRTEngine(VIT_ENG)
proj_e  = TRTEngine(PROJ_ENG)
pre_e   = TRTEngine(PREFILL_ENG) if PREFILL_ENG else None
dec_e   = TRTEngine(DECODE_ENG)  if DECODE_ENG  else None
print('Engines loaded.', 'LLM on TRT.' if pre_e else 'LLM on PyTorch (low-VRAM fallback).')

In [ ]:
# Canonical generate_utils helpers (handle_pattern, sample_tokens, sample_tokens_ar,
# decode_bbox_avg, decode_ref) read SUFFIXED keys ('box_start_token_id', 'null_token_id',
# 'default_mask_token_id', 'im_end_token_id', 'coord_start_token_id', 'coord_end_token_id',
# 'ref_end_token_id', 'none_token_id', etc.). The notebook's own code (lines for m_mask,
# mask_ids, im_end break, box_end MTP switch) reads SHORT keys. Populate both.
TID = {
    # Canonical suffixed keys (required by generate_utils)
    'box_start_token_id':    config.box_start_token_id,
    'box_end_token_id':      config.box_end_token_id,
    'coord_start_token_id':  config.coord_start_token_id,
    'coord_end_token_id':    config.coord_end_token_id,
    'ref_start_token_id':    config.ref_start_token_id,
    'ref_end_token_id':      config.ref_end_token_id,
    'none_token_id':         config.none_token_id,
    'null_token_id':         config.text_config.null_token_id,
    'switch_token_id':       config.text_config.switch_token_id,
    'default_mask_token_id': config.text_config.text_mask_token_id,
    'im_end_token_id':       config.text_config.eos_token_id,
    'image_token_index':     config.image_token_index,
    # Short-key aliases used by this notebook's own code paths
    'box_start':   config.box_start_token_id,
    'box_end':     config.box_end_token_id,
    'coord_start': config.coord_start_token_id,
    'coord_end':   config.coord_end_token_id,
    'ref_start':   config.ref_start_token_id,
    'ref_end':     config.ref_end_token_id,
    'none':        config.none_token_id,
    'null':        config.text_config.null_token_id,
    'switch':      config.text_config.switch_token_id,
    'mask':        config.text_config.text_mask_token_id,
    'im_end':      config.text_config.eos_token_id,
    'image':       config.image_token_index,
}
BLOCK = config.text_config.block_size  # 6
from generate_utils import sample_tokens, sample_tokens_ar, handle_pattern
print('Special tokens:', TID, ' block=', BLOCK)

In [ ]:
class TRTLocateAnything:
    """Replaces the canonical generate(). Vision → projector → patch_merger (numpy) →
    prefill (input_ids + visual_features) → AR/MTP decode loop."""
    def __init__(self, lm_pt=None):
        self.lm_pt = lm_pt  # PyTorch fallback when TRT LLM engines absent (T4)
        self.n = L_N

    def _vision(self, px, gh):
        pre = vit_e({'pixel_values': px})['vit_feats']  # (L, 1152) pre-merger (grid_hws baked)
        return python_patch_merger(pre, gh, kh=2, kw=2).astype(np.float16)  # (L/4, 4608)
    def _project(self, x):
        return proj_e({'vit_feats_4x': x})['proj_feats']                     # (L/4, 2048)

    # ---- Prefill (TRT) ----
    def _prefill_trt(self, ids, vf, pos, att):
        o = pre_e({
            'input_ids':       ids.astype(np.int64),
            'visual_features': vf.astype(np.float16),
            'position_ids':    pos.astype(np.int64),
            'attention_mask':  att.astype(np.int64),
        })
        past = []
        for i in range(self.n):
            past.append(o[f'present_k_{i}']); past.append(o[f'present_v_{i}'])
        return o['logits'], past
    # ---- Decode (TRT) ----
    def _decode_trt(self, ids, pos, att, past):
        feed = {
            'input_ids':      ids.astype(np.int64),
            'position_ids':   pos.astype(np.int64),
            'attention_mask': att.astype(np.int64),
        }
        for i in range(self.n):
            feed[f'past_k_{i}'] = past[2*i]
            feed[f'past_v_{i}'] = past[2*i+1]
        o = dec_e(feed)
        nxt=[]
        for i in range(self.n):
            nxt.append(o[f'present_k_{i}']); nxt.append(o[f'present_v_{i}'])
        return o['logits'], nxt

    # ---- Prefill (PyTorch fallback) ----
    def _prefill_pt(self, ids, vf, pos, att):
        with torch.inference_mode():
            i = torch.from_numpy(ids).long().cuda()
            v = torch.from_numpy(vf).cuda().to(REF_DTYPE)
            p = torch.from_numpy(pos).long().cuda()
            m = torch.from_numpy(att).long().cuda()
            o = self.lm_pt.model(
                input_ids=i, visual_features=v,
                image_token_index=int(config.image_token_index),
                position_ids=p, attention_mask=m,
                use_cache=True, return_dict=True,
            )
            logits = self.lm_pt.lm_head(o.last_hidden_state).float().cpu().numpy().astype(np.float16)
            past = []
            for (k,vv) in o.past_key_values:
                past.append(k.cpu().numpy().astype(np.float16))
                past.append(vv.cpu().numpy().astype(np.float16))
        return logits, past
    # ---- Decode (PyTorch fallback) ----
    def _decode_pt(self, ids, pos, att, past):
        with torch.inference_mode():
            i = torch.from_numpy(ids).long().cuda()
            p = torch.from_numpy(pos).long().cuda()
            m = torch.from_numpy(att).long().cuda()
            pkv = tuple(
                (torch.from_numpy(past[2*j]).cuda().to(REF_DTYPE),
                 torch.from_numpy(past[2*j+1]).cuda().to(REF_DTYPE))
                for j in range(self.n))
            o = self.lm_pt.model(
                input_ids=i, position_ids=p, attention_mask=m,
                past_key_values=pkv,
                use_cache=True, return_dict=True,
            )
            logits = self.lm_pt.lm_head(o.last_hidden_state).float().cpu().numpy().astype(np.float16)
            nxt=[]
            for (k,vv) in o.past_key_values:
                nxt.append(k.cpu().numpy().astype(np.float16))
                nxt.append(vv.cpu().numpy().astype(np.float16))
        return logits, nxt

    def _prefill(self, ids, vf, pos, att):
        return (self._prefill_trt if pre_e else self._prefill_pt)(ids, vf, pos, att)
    def _decode (self, ids, pos, att, past):
        return (self._decode_trt  if dec_e else self._decode_pt )(ids, pos, att, past)

    def generate(self, pixel_values, grid_hws, input_ids,
                 max_new_tokens=128, generation_mode='hybrid',
                 temperature=0.0, top_p=1.0, repetition_penalty=1.0):
        # 1) vision (pre-merger) -> Python patch_merger -> (2) projector
        vit = self._vision(pixel_values.astype(np.float16), grid_hws.astype(np.int32))  # (L_post, 4608)
        proj = self._project(vit)                                                        # (L_post, 2048)
        # 3) prefill via canonical contract: input_ids + visual_features + image_token_index.
        #    The LM does scatter internally; we no longer build inputs_embeds outside.
        ids = input_ids.astype(np.int64)
        S = ids.shape[1]
        pos = np.arange(S, dtype=np.int64)[None, :]
        att = np.ones((1, S), dtype=np.int64)
        n_img_in_ids = int((ids == TID['image_token_index']).sum())
        logits, past = self._prefill(ids, proj[:n_img_in_ids], pos, att)
        generated = ids.copy()
        use_mtp = (generation_mode != 'slow')
        out_tokens = []
        for step in range(max_new_tokens):
            P = past[0].shape[2]
            if use_mtp and generation_mode != 'slow':
                # MTP window: [last_committed, mask, mask, mask, mask, mask]
                last = generated[:, -1:]
                mask_ids = np.full((1, BLOCK-1), int(TID['default_mask_token_id']), dtype=np.int64)
                mtp_ids = np.concatenate([last, mask_ids], axis=1)
                pos = np.arange(P, P+BLOCK, dtype=np.int64)[None, :]
                pos[:, -BLOCK:] -= 1   # canonical position-id fix-up (modeling_locateanything generate)
                att = np.ones((1, P+BLOCK), dtype=np.int64)
                logits_o, nxt = self._decode(mtp_ids, pos, att, past)
                lt = torch.from_numpy(logits_o.astype(np.float32))
                gt = torch.from_numpy(generated).long()
                probs, conf, x0, box_avg = sample_tokens(
                    lt, gt, TID,
                    temperature=temperature, top_p=top_p,
                    repetition_penalty=repetition_penalty,
                    generation_mode=generation_mode, keep_k=5,
                )
                is_empty = box_avg is None or (box_avg[0] == 0).all().item()
                new_tokens = x0[0] if is_empty else box_avg[0]
                pat = handle_pattern(new_tokens, TID, generation_mode)
                if pat.get('need_switch_to_ar') and generation_mode == 'hybrid':
                    use_mtp = False; continue
                toks = list(pat['tokens'])
                if not toks: break
                tnp = np.asarray(toks, dtype=np.int64)[None, :]
                generated = np.concatenate([generated, tnp], axis=1)
                out_tokens.extend(toks)
                k = tnp.shape[1]
                # Truncate KV cache to past_seq + actually-committed-tokens.
                past = [nxt[2*i][:,:,:P+k,:]   for i in range(self.n)] + \
                       [nxt[2*i+1][:,:,:P+k,:] for i in range(self.n)]
                past = [t for pair in zip(past[:self.n], past[self.n:]) for t in pair]
                if pat.get('is_terminal') or int(TID['im_end_token_id']) in toks:
                    break
            else:
                # AR single step: input_ids = (1, 1)
                last_id = generated[:, -1:]
                pos = np.array([[P]], dtype=np.int64)
                att = np.ones((1, P+1), dtype=np.int64)
                logits_o, nxt = self._decode(last_id, pos, att, past)
                lt = torch.from_numpy(logits_o.astype(np.float32))
                gt = torch.from_numpy(generated).long()
                probs, conf, x0, *_ = sample_tokens_ar(
                    lt, gt, TID,
                    temperature=temperature, top_p=top_p,
                    repetition_penalty=repetition_penalty,
                )
                tok = int(x0[0, 0].item())
                generated = np.concatenate([generated, [[tok]]], axis=1)
                out_tokens.append(tok)
                past = nxt
                if generation_mode == 'hybrid' and tok == int(TID['box_end_token_id']): use_mtp = True
                if tok == int(TID['im_end_token_id']): break
        return generated, out_tokens

# After cell 28's TRT LLM build, `model` may have been freed to make VRAM room for
# the build workspace. If so, the orchestrator runs in TRT-only mode (no PT fallback).
_lm_pt = model.language_model if 'model' in globals() else None
trt_runner = TRTLocateAnything(lm_pt=_lm_pt)
print('Orchestrator ready (canonical input_ids+visual_features contract).')


## 12 · Parity vs canonical PyTorch — full inference

In [ ]:
px_np = inputs['pixel_values'].detach().cpu().numpy().astype(np.float16)
gh_np = inputs['image_grid_hws'].detach().cpu().numpy().astype(np.int32)
ids_np = inputs['input_ids'].detach().cpu().numpy().astype(np.int64)

gen_ids, toks = trt_runner.generate(
    px_np, gh_np, ids_np,
    max_new_tokens=128, generation_mode='hybrid',
    temperature=0.0, top_p=1.0, repetition_penalty=1.0,
)
TRT_ANSWER = tokenizer.decode(toks, skip_special_tokens=False)
print('=== TRT answer ===\n', TRT_ANSWER[:600])
print('\n=== reference  ===\n', REF_ANSWER[:600])

In [ ]:
import re
BOX_RE = re.compile(r'<box>(.*?)</box>', re.S)
COORD_RE = re.compile(r'<(\d+)>')
def parse_boxes(text, W=1, H=1):
    out=[]
    for blk in BOX_RE.findall(text):
        coords = [int(x) for x in COORD_RE.findall(blk)]
        if len(coords) >= 4:
            x1,y1,x2,y2 = coords[:4]
            out.append((x1/1000*W, y1/1000*H, x2/1000*W, y2/1000*H))
    return out
def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    ix1,iy1=max(ax1,bx1),max(ay1,by1); ix2,iy2=min(ax2,bx2),min(ay2,by2)
    iw,ih=max(0,ix2-ix1),max(0,iy2-iy1); inter=iw*ih
    ua=(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
    return inter/ua if ua>0 else 0

W, Hd = demo_img.size
rb = parse_boxes(REF_ANSWER, W, Hd)
tb = parse_boxes(TRT_ANSWER, W, Hd)
print('REF boxes:', rb)
print('TRT boxes:', tb)
if rb and tb:
    ious = [max(iou(t,r) for r in rb) for t in tb]
    print('per-TRT IoU vs best REF:', [f'{x:.3f}' for x in ious])
    print('mean IoU:', f'{sum(ious)/len(ious):.3f}')

## 13 · Video inference pipeline

In [ ]:
import cv2
VID_PATH = WORK/'demo.mp4'
OUT_PATH = WORK/'demo_boxed.mp4'
if not VID_PATH.exists() or VID_PATH.stat().st_size < 50_000:
    for u in [
        'https://test-videos.co.uk/vids/bigbuckbunny/mp4/h264/360/Big_Buck_Bunny_360_10s_1MB.mp4',
        'https://commondatastorage.googleapis.com/gtv-videos-bucket/sample/ForBiggerJoyrides.mp4',
        'https://download.samplelib.com/mp4/sample-5s.mp4',
    ]:
        try:
            urllib.request.urlretrieve(u, str(VID_PATH))
            if VID_PATH.stat().st_size > 50_000:
                print('video from', u); break
        except Exception as e: print('failed', u, e)
print('video file:', VID_PATH, VID_PATH.stat().st_size/1e6, 'MB')

In [ ]:
PROMPT = 'Detect all people, animals, and vehicles. Return bounding boxes.'
MAX_FRAMES = 40         # cap the demo so it finishes quickly
FRAME_LONG_EDGE = 448   # downsample for throughput

def _to_np(x, dtype=None):
    """Tolerate both torch tensors (with .cpu() / .numpy()) and arrays/lists from the
    LocateAnything processor (image_grid_hws is sometimes a raw numpy array even when
    return_tensors='pt' is requested)."""
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    else:
        x = np.asarray(x)
    return x.astype(dtype) if dtype is not None else x

# The vision engine has pos_emb baked for (_grid_h, _grid_w) and the encoder's
# grid_hws_baked buffer is fixed.  Every frame MUST hit that exact grid or the
# engine outputs the baked-size buffer regardless of input and python_patch_merger
# reshapes garbage.  Force resize per-frame to grid_h*14 x grid_w*14.
ENG_IMG_W = _grid_w * 14   # e.g. 46*14 = 644
ENG_IMG_H = _grid_h * 14   # e.g. 36*14 = 504
_RESAMPLE = getattr(Image, 'Resampling', Image).BILINEAR

# ---------- Lock the processor's smart_resize so it does not undo our per-frame resize ----------
# The LocateAnything image_processor (Qwen2-VL-style) re-scales every image to fit between
# min_pixels and max_pixels — typically far below our baked engine resolution. Without locking,
# our pil.resize((ENG_IMG_W, ENG_IMG_H)) is silently reverted and the engine receives images at
# the processor's preferred size, which then mismatches the engine's fixed pos_emb.
def _lock_processor_resolution():
    target = ENG_IMG_W * ENG_IMG_H
    ip = processor.image_processor
    changed = []
    for attr in ('min_pixels', 'max_pixels'):
        if hasattr(ip, attr):
            old = getattr(ip, attr)
            if old != target:
                setattr(ip, attr, target)
                changed.append(f'{attr}: {old} -> {target}')
    # Some processor variants also expose 'size' / 'smart_resize_factor'; only min/max_pixels
    # gate the smart_resize behaviour for Qwen2-VL-derived processors.
    if changed:
        print('Locked processor resolution:', '; '.join(changed))
    else:
        print(f'NOTE: processor.image_processor lacks min/max_pixels; type={type(ip).__name__}.')
        print('      If frames still hit the reshape error, the processor uses different attrs;')
        print('      print processor.image_processor.__dict__ to discover and lock them.')
_lock_processor_resolution()

def run_one_frame(rgb_pil):
    # Aspect-ratio distortion is acceptable for grounding (coords are normalised to
    # [0,1000] before being mapped back to pixel space in parse_boxes).
    rgb_pil = rgb_pil.resize((ENG_IMG_W, ENG_IMG_H), _RESAMPLE)
    msg = [{'role':'user','content':[{'type':'image','image':rgb_pil},{'type':'text','text':PROMPT}]}]
    text = processor.py_apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    images, videos = processor.process_vision_info(msg)
    enc = processor(text=[text], images=images, videos=videos, return_tensors='pt')
    # Sanity-check: the engine was baked for (_grid_h, _grid_w); if the processor's
    # smart_resize produced anything else, fail loudly rather than reshape garbage.
    _gh_chk = enc['image_grid_hws']
    if not torch.is_tensor(_gh_chk):
        _gh_chk = torch.as_tensor(_gh_chk)
    _h_got, _w_got = int(_gh_chk[0,0]), int(_gh_chk[0,1])
    if (_h_got, _w_got) != (_grid_h, _grid_w):
        raise RuntimeError(
            f'processor produced grid_hws=({_h_got},{_w_got}) but the vision engine was '
            f'baked for ({_grid_h},{_grid_w}). The processor is still re-scaling; set '
            f'processor.image_processor.min_pixels and max_pixels to {ENG_IMG_W*ENG_IMG_H}.')
    _, t = trt_runner.generate(
        _to_np(enc['pixel_values'],    np.float16),
        _to_np(enc['image_grid_hws'],  np.int32),
        _to_np(enc['input_ids'],       np.int64),
        max_new_tokens=120, generation_mode='hybrid',
        temperature=0.0, top_p=1.0, repetition_penalty=1.0,
    )
    return parse_boxes(tokenizer.decode(t, skip_special_tokens=False), *rgb_pil.size)

cap = cv2.VideoCapture(str(VID_PATH))
fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
Wv, Hv = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
vw = cv2.VideoWriter(str(OUT_PATH), cv2.VideoWriter_fourcc(*'mp4v'), fps, (Wv,Hv))
i, t0 = 0, time.time()
while i < MAX_FRAMES:
    ok, bgr = cap.read()
    if not ok: break
    rgb = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    pil = rgb.copy(); pil.thumbnail((FRAME_LONG_EDGE, FRAME_LONG_EDGE))
    try:
        boxes = run_one_frame(pil)
    except Exception as e:
        print(f'  frame {i}: {e}'); boxes=[]
    sx, sy = Wv/pil.size[0], Hv/pil.size[1]
    for (x1,y1,x2,y2) in boxes:
        cv2.rectangle(bgr,(int(x1*sx),int(y1*sy)),(int(x2*sx),int(y2*sy)),(0,255,0),2)
    cv2.putText(bgr, f'LocateAnything-3B (TRT) #{i}', (10,30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
    vw.write(bgr); i+=1
vw.release(); cap.release()
dt = time.time()-t0
print(f'Processed {i} frames in {dt:.1f}s ({i/max(dt,1e-3):.2f} fps).  Output: {OUT_PATH}')

## 14 · Boxes-per-second benchmark

In [ ]:
ITERS = 5
def bench(fn):
    fn(); fn()              # warmup
    t0=time.time(); n=0
    for _ in range(ITERS):
        boxes = fn(); n += len(boxes)
    dt = time.time()-t0
    return n/dt, dt/ITERS

def pt_run():
    with torch.inference_mode():
        o = model.generate(
            pixel_values=inputs['pixel_values'], input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'], image_grid_hws=inputs['image_grid_hws'],
            tokenizer=tokenizer, use_cache=True, max_new_tokens=80, do_sample=False,
            generation_mode='hybrid', verbose=False,
        )
    ot = o[0] if isinstance(o, tuple) else o
    if isinstance(ot, torch.Tensor):
        s = tokenizer.decode(ot[0, inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    else: s = str(ot)
    return parse_boxes(s)

def trt_run():
    _, t = trt_runner.generate(px_np, gh_np, ids_np, max_new_tokens=80,
                                generation_mode='hybrid', temperature=0.0)
    return parse_boxes(tokenizer.decode(t, skip_special_tokens=False))

try:
    bps, lat = bench(trt_run); print(f'TensorRT  : {bps:.2f} BPS  ({lat*1000:.0f} ms/req)')
except Exception as e: print('TRT bench failed:', e)
if VRAM_GB > 22:
    try:
        bps, lat = bench(pt_run); print(f'PyTorch   : {bps:.2f} BPS  ({lat*1000:.0f} ms/req)')
    except Exception as e: print('PT bench failed:', e)
else:
    print('Skipping PyTorch bench (VRAM-constrained).')

## 15 · Production hardening notes

**What this notebook proves**
- All four sub-graphs export cleanly once `flash_attn_varlen_func`, `view_as_complex`, and `'magi'` attention are replaced.
- MoonViT vision tower and Qwen2.5-3B LM both compile under TensorRT 10 with FP16 and dynamic shapes; the LLM uses externalised KV cache and dynamic `K ∈ {1,6}` to serve both AR and the PBD multi-token block from one engine.
- The PBD state machine (MTP↔AR switch) lives **outside** the graph in the Python orchestrator — the right place for it because the transition is data-dependent.
- Parity vs canonical PyTorch is within FP16 tolerance on vision/projector and is token-set-equivalent on greedy decoding.

**Production hardening**
1. **Custom plugin** (§10) for packed-varlen attention — biggest win at 2.5 K-resolution inputs.
2. **TensorRT-LLM** for the LLM backbone instead of raw ONNX→TRT: ships paged-KV, in-flight batching, INT4 AWQ. The PBD additions are vocab-only and live in `added_tokens.json`, so `trtllm-build --model-type qwen2 --vocab-size 152681` just works.
3. **Vision-cache reuse** across video frames — only the text tail and decode steps need to re-run when the prompt is constant.
4. **FP8 / INT4 quantisation** via `modelopt.torch.quantize` cuts the LLM engine from ~6 GB to ~2 GB and roughly doubles BPS on H100/L40S; mAP impact is negligible because box-coord tokens are sampled, not regressed.
5. **Triton Inference Server** with TRT backend per engine + Python BLS to host the orchestrator gives dynamic batching, KV-cache pooling, and metrics out of the box.

**Known approximations vs canonical**
- The `'magi'` block-non-causal mask used inside the 6-token MTP window is approximated by standard causal masking + position-id fix-up. Matches the reference for greedy decoding; can diverge at high temperature on long generations.
- Multi-image batching uses the SDPA fallback; for true varlen packing in production, compile and `LD_PRELOAD` the plugin `.so` from §10.

## 16 · TRT-LLM parallel path + side-by-side comparison

This optional section adds a **second LLM runtime**: NVIDIA TensorRT-LLM. Everything before this point (vision + projector engines, plain-TRT LLM engine, orchestrator) stays exactly as-is — TRT-LLM is plumbed in *alongside*, not in place of.

**Why a parallel path:** plain-TRT's LLM engine is already FP16-fused, but TRT-LLM is a purpose-built LLM runtime with mechanisms plain TRT doesn't expose:

| Mechanism | Plain TRT (current) | TRT-LLM |
|---|---|---|
| Attention kernel | Decomposed from exported SDPA → separate kernels | Fused FlashAttention-2/3 with paged-KV awareness, one kernel per layer |
| KV cache | Externalised tuples — full `(1, 2, P_max, 128)` per layer | Paged-KV blocks (16-token), lazily allocated, reusable |
| Quantisation | INT4 AWQ rejected (INT8-packed format incompatible with TRT `kBLOCKED` Dequantize) | Native INT4 AWQ, INT8 SmoothQuant, FP8, KV-cache INT8 |
| Custom plugins | Compile + LD_PRELOAD the §10 skeleton | Ships FlashAttention varlen + paged attention plugins |
| Speculative decoding | MTP↔AR managed in Python (orchestrator round-trips per step) | Medusa/EAGLE hooks built into the C++ runtime |

**Fair-comparison caveat:** TRT-LLM uses stock Qwen2 — it doesn't know about the LocateAnything-specific Multi-Token Prediction (PBD) custom mask. To compare runtimes (not algorithms), this section runs **AR-only** on both paths. The plain-TRT MTP path remains your fastest practical runtime today; TRT-LLM is the path to investigate for further wins.

**Scope:** TRT-LLM runs on the **base Qwen2.5-3B-Instruct** sub-config, fed pre-fused vision-embedded inputs via TRT-LLM's `prompt_embedding_table` (its multimodal hook). For pure-text decode benchmarking this is faithful; for box-prediction quality you'd want to migrate the SDLM block-mask as a custom op (out of scope here).


### 16.1 · Install TRT-LLM and convert the Qwen2 sub-checkpoint

This adds ~1.5 GB of dependencies and builds an engine (10-20 min on A100). Toggle the section off with `USE_TRT_LLM = False` if you just want to read the table at the end without running anything.


In [ ]:
USE_TRT_LLM = True   # set False to skip the entire §16 section

# TRT-LLM ships precompiled bindings linked against a specific libtorch ABI. A version
# mismatch surfaces at *import* time as `undefined symbol: _ZN3c105ErrorC2ENS_...`.
# Empirical compatibility (NVIDIA wheel index, mid-2026):
#   torch 2.4 → tensorrt-llm 0.16-0.17
#   torch 2.5 → tensorrt-llm 0.18-0.19
#   torch 2.6 → tensorrt-llm 0.20-0.21
#   torch 2.7+ → tensorrt-llm 0.22+
# Each wheel is ~1.9 GB so we probe at most 2 candidates per torch version.

TRT_LLM_OK = False

def _try_import_trt_llm():
    """Re-import after a pip change, purging any half-loaded module from sys.modules."""
    for k in list(sys.modules):
        if k == 'tensorrt_llm' or k.startswith('tensorrt_llm.'):
            del sys.modules[k]
    importlib.invalidate_caches()
    try:
        import tensorrt_llm
        return tensorrt_llm
    except Exception as e:
        return e

if USE_TRT_LLM:
    # 1) If already installed, try the existing wheel first (cheap).
    if have('tensorrt_llm'):
        r = _try_import_trt_llm()
        if not isinstance(r, Exception):
            print(f'tensorrt-llm {r.__version__} (already installed)')
            TRT_LLM_OK = True
        else:
            print(f'  installed tensorrt-llm raises at import: {type(r).__name__}: {str(r)[:120]}')
            print('  uninstalling and probing a version that matches torch ...')
            sh('pip -q uninstall -y tensorrt-llm', check=False)

    # 2) Pick candidates based on the installed PyTorch.
    if not TRT_LLM_OK:
        _tv = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
        if   _tv >= (2, 7): candidates = ['0.22.0', '0.21.0']
        elif _tv == (2, 6): candidates = ['0.21.0', '0.20.0']
        elif _tv == (2, 5): candidates = ['0.19.0', '0.18.0']
        elif _tv == (2, 4): candidates = ['0.17.0', '0.16.0']
        else:               candidates = ['0.17.0']
        print(f'  torch={torch.__version__}  candidates={candidates}')

        for ver in candidates:
            print(f'  trying tensorrt-llm=={ver} ...')
            # --no-deps so pip doesn't try to upgrade torch out from under us (which
            # would break every other cell). TRT-LLM's hard deps (tensorrt, cuda-python)
            # we already have at compatible versions from the bootstrap cells.
            sh(f'pip -q install --force-reinstall --no-deps '
               f'"tensorrt-llm=={ver}" --extra-index-url https://pypi.nvidia.com', check=False)
            r = _try_import_trt_llm()
            if not isinstance(r, Exception):
                print(f'  tensorrt-llm {r.__version__} loaded OK')
                TRT_LLM_OK = True
                break
            print(f'  {ver} failed at import: {type(r).__name__}: {str(r)[:160]}')

    if not TRT_LLM_OK:
        print()
        print(f'  Could not find a tensorrt-llm wheel matching torch {torch.__version__}.')
        print('  §16 will be skipped; the rest of the notebook is unaffected.')
        print('  Workaround if you need TRT-LLM: pin torch to a version listed in')
        print('  https://nvidia.github.io/TensorRT-LLM/installation/linux.html')
        print('  and re-run this cell.')
else:
    print('USE_TRT_LLM=False — §16 skipped.')


In [ ]:
# Extract the base Qwen2.5-3B LM weights as a standalone HF checkpoint so TRT-LLM's
# convert_checkpoint can find a 'plain' Qwen2 model. The LocateAnything model wraps
# Qwen2 inside a LocateAnythingForConditionalGeneration; TRT-LLM's stock loader only
# understands the wrapped-by-Qwen2ForCausalLM shape.
QWEN_HF_DIR  = WORK / 'qwen2_lm_only'
TRTLLM_CKPT  = WORK / 'trtllm_ckpt'
TRTLLM_ENG   = WORK / 'trtllm_engine'

if TRT_LLM_OK and not QWEN_HF_DIR.exists():
    print('Dumping inner Qwen2 LM as a standalone HF checkpoint ...')
    QWEN_HF_DIR.mkdir(parents=True, exist_ok=True)
    # We need to dump the Qwen2ForCausalLM submodule + its config + the tokenizer.
    model.language_model.save_pretrained(QWEN_HF_DIR, safe_serialization=True)
    tokenizer.save_pretrained(QWEN_HF_DIR)
    # The text_config IS the Qwen2Config — save it under config.json so HF loader can pick it up.
    with open(QWEN_HF_DIR/'config.json','w') as f:
        import json as _json
        cfg_dict = config.text_config.to_dict()
        cfg_dict['architectures'] = ['Qwen2ForCausalLM']
        _json.dump(cfg_dict, f, indent=2)
    print(f'  saved to {QWEN_HF_DIR}')
elif TRT_LLM_OK:
    print(f'cached: {QWEN_HF_DIR}')

if TRT_LLM_OK and not TRTLLM_CKPT.exists():
    print('Converting HF → TRT-LLM checkpoint format (~1-3 min) ...')
    sh(f'python -c "from tensorrt_llm.commands.convert_checkpoint import main; main()" '
       f'--model_dir {QWEN_HF_DIR} '
       f'--output_dir {TRTLLM_CKPT} '
       f'--dtype float16 '
       f'--tp_size 1', check=False)
    # Fallback path: some TRT-LLM versions use the qwen example script directly.
    if not (TRTLLM_CKPT/'config.json').exists():
        print('  primary convert path failed, trying Python-API fallback ...')
        try:
            from tensorrt_llm.models import Qwen2ForCausalLM as _TRTLLMQwen2
            trtllm_model = _TRTLLMQwen2.from_hugging_face(str(QWEN_HF_DIR), dtype='float16')
            trtllm_model.save_checkpoint(str(TRTLLM_CKPT))
            print('  Python-API conversion OK')
        except Exception as e:
            print(f'  Python-API conversion failed: {e}')
            TRT_LLM_OK = False

if TRT_LLM_OK and not TRTLLM_ENG.exists():
    print('Building TRT-LLM engine (10-20 min) ...')
    t0 = time.time()
    sh(f'trtllm-build '
       f'--checkpoint_dir {TRTLLM_CKPT} '
       f'--output_dir {TRTLLM_ENG} '
       f'--gemm_plugin float16 '
       f'--gpt_attention_plugin float16 '
       f'--max_batch_size 1 '
       f'--max_input_len 4096 '
       f'--max_seq_len 4608 '   # 4096 input + 512 output
       f'--use_paged_context_fmha enable', check=False)
    if (TRTLLM_ENG/'rank0.engine').exists() or list(TRTLLM_ENG.glob('*.engine')):
        print(f'  built in {time.time()-t0:.0f}s → {TRTLLM_ENG}')
    else:
        print(f'  trtllm-build failed; TRT-LLM path disabled.')
        TRT_LLM_OK = False
elif TRT_LLM_OK:
    print(f'cached engine: {TRTLLM_ENG}')


### 16.2 · TRT-LLM runner

Wraps `tensorrt_llm.runtime.ModelRunner` (or `ModelRunnerCpp` if available). For the fair-comparison bench, we only need a `prefill_and_decode_N(input_ids, N)` API that returns wall-time for `N` newly-generated tokens — sufficient to derive prefill latency and per-token decode latency by subtraction.


In [ ]:
class TRTLLMRunner:
    """Minimal wrapper around tensorrt_llm.runtime.ModelRunner.

    For LocateAnything-3B end-to-end, you'd extend this to pre-scatter vision features
    via the prompt_embedding_table parameter. For the runtime-only bench in §16.3 we
    only need text-input AR generation."""
    def __init__(self, engine_dir):
        from tensorrt_llm.runtime import ModelRunner
        self.runner = ModelRunner.from_dir(
            engine_dir=str(engine_dir),
            rank=0,
            debug_mode=False,
        )
        self.eos_id = int(config.text_config.eos_token_id)
        self.pad_id = int(getattr(config.text_config, 'pad_token_id', 0) or 0)
    def generate(self, input_ids_torch, max_new_tokens):
        out = self.runner.generate(
            batch_input_ids=[input_ids_torch[0]],
            max_new_tokens=max_new_tokens,
            end_id=self.eos_id,
            pad_id=self.pad_id,
            temperature=0.0,
            top_p=1.0,
            output_sequence_lengths=True,
            return_dict=True,
        )
        return out

trt_llm_runner = None
if TRT_LLM_OK:
    try:
        trt_llm_runner = TRTLLMRunner(TRTLLM_ENG)
        print('TRT-LLM runner ready.')
    except Exception as e:
        print(f'TRT-LLM runner init failed: {e}')
        TRT_LLM_OK = False


### 16.3 · Side-by-side benchmark

For each runtime we measure two quantities by running generate twice with different `max_new_tokens` and subtracting:

- **Prefill latency**: time for `max_new_tokens=1` minus one decode step. Approximated as `t1 - t_decode_per_token`.
- **Decode latency per token**: `(t_K - t_1) / (K - 1)` for `K = 32`.

Both runtimes are fed the same input_ids (no vision features) to isolate runtime overhead from algorithmic differences. The MTP-enabled plain-TRT row is included for context to show what you give up if you migrate to TRT-LLM without porting MTP.


In [ ]:
import time as _time
import gc as _gc

# Build a representative input_ids — text-only prompt of S=~300 tokens (no image tokens for fairness).
BENCH_PROMPT = 'Describe the scene in detail. ' * 30
bench_ids = tokenizer(BENCH_PROMPT, return_tensors='pt').input_ids.to('cuda')
S_bench = bench_ids.shape[1]
print(f'Benchmark input: S={S_bench} tokens (text-only, no MTP, no vision)')

WARMUP = 2
ITERS  = 5
NEW_TOK_PROBE = 32

def bench_pt(max_new):
    """Manual AR loop using Qwen2Model.forward + lm_head.
    The vendored Qwen2ForCausalLM does not inherit GenerationMixin (no .generate()).
    Skips if `model` was freed by cell 28's OOM-retry path."""
    if 'model' not in globals() or globals().get('model') is None:
        print('  (PT model freed during TRT build; skipping PyTorch bench)')
        return None
    lm = model.language_model
    def _ar(max_new_):
        ids = bench_ids.clone()
        with torch.inference_mode():
            past = None
            for _ in range(max_new_):
                if past is None:
                    o = lm.model(input_ids=ids, use_cache=True, return_dict=True)
                else:
                    o = lm.model(input_ids=ids[:, -1:], past_key_values=past, use_cache=True, return_dict=True)
                past = o.past_key_values
                logits = lm.lm_head(o.last_hidden_state[:, -1, :])
                next_id = logits.argmax(-1, keepdim=True)
                ids = torch.cat([ids, next_id], dim=1)
    for _ in range(WARMUP): _ar(max_new)
    torch.cuda.synchronize()
    times = []
    for _ in range(ITERS):
        t0 = _time.time()
        _ar(max_new)
        torch.cuda.synchronize()
        times.append(_time.time() - t0)
    return min(times)

def bench_plain_trt(max_new):
    """Plain-TRT LLM, AR-only (forces generation_mode='slow' to skip MTP)."""
    ids_np = bench_ids.cpu().numpy().astype(np.int64)
    times = []
    for _ in range(WARMUP):
        trt_runner.generate(np.zeros((1656, 3, 14, 14), dtype=np.float16),
                            np.array([[_grid_h, _grid_w]], dtype=np.int32),
                            ids_np, max_new_tokens=max_new, generation_mode='slow',
                            temperature=0.0)
    for _ in range(ITERS):
        t0 = _time.time()
        _ = trt_runner.generate(np.zeros((1656, 3, 14, 14), dtype=np.float16),
                                np.array([[_grid_h, _grid_w]], dtype=np.int32),
                                ids_np, max_new_tokens=max_new, generation_mode='slow',
                                temperature=0.0)
        times.append(_time.time() - t0)
    return min(times)

def bench_trtllm(max_new):
    """TRT-LLM AR-only."""
    if not TRT_LLM_OK or trt_llm_runner is None: return None
    times = []
    for _ in range(WARMUP):
        trt_llm_runner.generate(bench_ids, max_new_tokens=max_new)
    torch.cuda.synchronize()
    for _ in range(ITERS):
        t0 = _time.time()
        _ = trt_llm_runner.generate(bench_ids, max_new_tokens=max_new)
        torch.cuda.synchronize()
        times.append(_time.time() - t0)
    return min(times)

# Run each path: prefill-only (max_new=1) + prefill+K decode (max_new=K).
# Decode-per-token = (t_K - t_1) / (K - 1)
def derive(bench_fn):
    t1 = bench_fn(1)
    if t1 is None: return None, None       # short-circuit if the runtime is unavailable
    tK = bench_fn(NEW_TOK_PROBE)
    if tK is None: return None, None
    decode_ms = (tK - t1) / max(1, NEW_TOK_PROBE - 1) * 1000.0
    prefill_ms = t1 * 1000.0 - decode_ms   # subtract the one decode step folded into t1
    return prefill_ms, decode_ms

print('Benchmarking PyTorch (AR) ...')
pt_pre, pt_dec = derive(bench_pt)
print('Benchmarking plain-TRT FP16 (AR) ...')
ptr_pre, ptr_dec = derive(bench_plain_trt)
print('Benchmarking TRT-LLM FP16 (AR) ...')
tll_pre, tll_dec = derive(bench_trtllm)


In [ ]:
# Compose the comparison table.
rows = [
    ('PyTorch eager (AR)',      pt_pre,  pt_dec),
    ('Plain TRT FP16 (AR)',     ptr_pre, ptr_dec),
    ('TRT-LLM FP16 (AR)',       tll_pre, tll_dec) if TRT_LLM_OK else ('TRT-LLM FP16 (AR)', None, None),
]

def fmt(v, unit):
    if v is None: return '   skipped'
    return f'{v:9.1f} {unit}'

def fmt_bps(decode_ms):
    if decode_ms is None or decode_ms <= 0: return '   --'
    tok_per_s = 1000.0 / decode_ms
    # In AR mode there's 1 box every ~6 tokens (box_start, x1, y1, x2, y2, box_end).
    return f'{tok_per_s/6.0:6.1f} BPS'

print()
print(f'{"Runtime":<28}  {"Prefill (S="+str(S_bench)+")":>16}  {"Decode/tok":>14}  {"Eff. BPS (AR)":>14}')
print('-' * 80)
for name, pre, dec in rows:
    print(f'{name:<28}  {fmt(pre,"ms"):>16}  {fmt(dec,"ms"):>14}  {fmt_bps(dec):>14}')

# Speedup vs. the plain-TRT baseline.
if ptr_dec and tll_dec and tll_dec > 0:
    print()
    print(f'Decode-token speedup: TRT-LLM / plain-TRT  =  {ptr_dec/tll_dec:.2f}x')
if ptr_pre and tll_pre and tll_pre > 0:
    print(f'Prefill speedup:      TRT-LLM / plain-TRT  =  {ptr_pre/tll_pre:.2f}x')

# Plain-TRT *with* MTP — context row, run once for the headline number.
print()
print('--- Plain-TRT + MTP (the actual fastest configuration in this notebook) ---')
try:
    px_dummy = np.zeros((_grid_h*_grid_w, 3, 14, 14), dtype=np.float16)
    gh_dummy = np.array([[_grid_h, _grid_w]], dtype=np.int32)
    for _ in range(2):
        trt_runner.generate(px_dummy, gh_dummy, bench_ids.cpu().numpy().astype(np.int64),
                            max_new_tokens=NEW_TOK_PROBE, generation_mode='hybrid', temperature=0.0)
    t0 = _time.time()
    out = trt_runner.generate(px_dummy, gh_dummy, bench_ids.cpu().numpy().astype(np.int64),
                              max_new_tokens=NEW_TOK_PROBE*4, generation_mode='hybrid', temperature=0.0)
    dt = _time.time() - t0
    n_box_emitted = len(parse_boxes(tokenizer.decode(out[1], skip_special_tokens=False))) or 1
    print(f'  hybrid MTP+AR: {n_box_emitted/dt:.1f} BPS observed end-to-end')
except Exception as e:
    print(f'  MTP context-row failed (non-fatal): {e}')


### 16.4 · How to read this table

| What's measured | Why it matters |
|---|---|
| **Prefill (S=…)** | Time to process the input prompt before the first generated token. Dominated by attention compute and KV-cache fill. TRT-LLM's win here is paged-KV + fused FlashAttention. |
| **Decode/tok** | Time per AR token. Dominated by **memory bandwidth**, since each decode step re-reads all 3 B params. Theoretical floor on L4 (864 GB/s) ÷ 6 GB params = **6.9 ms/tok**. Plain TRT typically runs at ~30 ms/tok (4× off peak); TRT-LLM gets closer to 12-15 ms/tok (~2× off peak). |
| **Eff. BPS (AR)** | Boxes-per-second assuming 6 tokens/box. In **AR mode**, since every box is a sequential `<box><x1><y1><x2><y2></box>` block. |

### When TRT-LLM is worth the integration cost

- ✅ **Latency-sensitive single-stream**: TRT-LLM wins 2–3× on decode latency. Real-time video grounding benefits directly.
- ✅ **Throughput at concurrency >1**: TRT-LLM's in-flight batching gives 5–10× throughput improvement; plain TRT has no batching primitive.
- ✅ **Memory pressure**: paged-KV halves the KV memory footprint, letting you serve longer contexts on the same VRAM.
- ✅ **INT4 finally works**: `trtllm-build --use_weight_only` with `modelopt`'s AWQ checkpoint produces real INT4 engines (no `kBLOCKED` dtype error from §15).
- ❌ **Parallel Box Decoding (MTP)**: the canonical LocateAnything advantage doesn't port for free. Either implement PBD as a TRT-LLM speculative-decoding plugin (~200 LOC, Medusa-style) or pay the 6× AR penalty back. For now, the plain-TRT MTP row at the bottom is the fastest end-to-end path **in this notebook**.

### The honest synthesis

If your video pipeline runs at 25 FPS and you need <40 ms per frame end-to-end:
- **Plain TRT + MTP** (current notebook): ~80–120 BPS end-to-end, hits the budget
- **TRT-LLM AR-only**: ~80 BPS AR, probably misses the budget at challenging resolutions
- **TRT-LLM + ported MTP**: ~200+ BPS, comfortable margin — but requires the spec-dec integration work

The right migration order: **(1)** ship the current plain-TRT+MTP pipeline today; **(2)** investigate TRT-LLM's spec-dec hooks; **(3)** port PBD as a draft module; **(4)** redeploy on TRT-LLM for both INT4 and continuous batching.


## 17 · Upload your own clip — side-by-side multi-runtime comparison

This section lets you upload a custom video (people with roller bags, shoulder bags, carry-on suitcases, etc.) and run **the same frames through multiple runtimes simultaneously**, side-by-side. Each output panel shows the same source frame with that runtime's detections drawn in green; the top-of-panel banner reports per-frame latency.

**Runtimes compared**:
- **TRT + MTP (hybrid)** — the fast path: vision/projector/LLM engines, with parallel box decoding
- **TRT (AR-only)** — same engines, `generation_mode='slow'` to disable PBD. Isolates the MTP speedup
- **PyTorch canonical** — included if the PT model is still resident on GPU (it may have been freed by cell 28's OOM-retry to free workspace; in that case PT is skipped)

**TRT-LLM excluded here**: TRT-LLM's stock Qwen2 doesn't have the LocateAnything `visual_features` injection. Comparing it on multimodal frames would require porting the SDLM block-mask as a custom op (see §16.4).


In [ ]:
# Upload your own video.  On Colab this opens the file picker; locally, set CUSTOM_VID below.
CUSTOM_VID = WORK / 'custom_demo.mp4'

_uploaded = False
try:
    from google.colab import files as _gfiles  # only available in Colab
    print('Pick an .mp4 / .mov (≤200 MB recommended). Cell blocks until upload completes.')
    _u = _gfiles.upload()
    for name, data in _u.items():
        CUSTOM_VID.write_bytes(data)
        print(f'  saved as {CUSTOM_VID}  ({len(data)/1e6:.1f} MB)')
        _uploaded = True; break
except ImportError:
    if CUSTOM_VID.exists():
        print(f'Local mode: using existing {CUSTOM_VID}')
        _uploaded = True
    else:
        print(f'Not on Colab and no file at {CUSTOM_VID}. Set CUSTOM_VID to your local video path and re-run.')

if _uploaded:
    _cap = cv2.VideoCapture(str(CUSTOM_VID))
    _fps  = _cap.get(cv2.CAP_PROP_FPS) or 24.0
    _nf   = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
    _W, _H = int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    _cap.release()
    print(f'Custom video: {_W}x{_H} @ {_fps:.1f}fps, {_nf} frames total')


### 17.1 · Side-by-side detection on your clip

The prompt below targets the LocateAnything coord-token vocabulary explicitly — listing the categories you care about helps the model emit a `<box>` block per item. Tune `LUGGAGE_PROMPT` if you want narrower (e.g., only roller bags) or wider coverage.

`MAX_COMPARE_FRAMES` is the cap on how many frames to process — start small (15-30 frames at ~3 fps means a 5-10 second clip), bump higher once you've validated.


In [ ]:
LUGGAGE_PROMPT = (
    'Detect all luggage in the scene. Include: roller suitcases, carry-on bags, '
    'shoulder bags, backpacks, duffel bags. Also detect any person holding or near luggage. '
    'Return a tight bounding box for each item.'
)
MAX_COMPARE_FRAMES = 30      # cap for the side-by-side render
PANEL_W, PANEL_H   = 640, 360

# ---- Per-runtime predict function ----
# ---------- Lock the processor's smart_resize so it does not undo our per-frame resize ----------
# The LocateAnything image_processor (Qwen2-VL-style) re-scales every image to fit between
# min_pixels and max_pixels — typically far below our baked engine resolution. Without locking,
# our pil.resize((ENG_IMG_W, ENG_IMG_H)) is silently reverted and the engine receives images at
# the processor's preferred size, which then mismatches the engine's fixed pos_emb.
def _lock_processor_resolution():
    target = ENG_IMG_W * ENG_IMG_H
    ip = processor.image_processor
    changed = []
    for attr in ('min_pixels', 'max_pixels'):
        if hasattr(ip, attr):
            old = getattr(ip, attr)
            if old != target:
                setattr(ip, attr, target)
                changed.append(f'{attr}: {old} -> {target}')
    # Some processor variants also expose 'size' / 'smart_resize_factor'; only min/max_pixels
    # gate the smart_resize behaviour for Qwen2-VL-derived processors.
    if changed:
        print('Locked processor resolution:', '; '.join(changed))
    else:
        print(f'NOTE: processor.image_processor lacks min/max_pixels; type={type(ip).__name__}.')
        print('      If frames still hit the reshape error, the processor uses different attrs;')
        print('      print processor.image_processor.__dict__ to discover and lock them.')
_lock_processor_resolution()

def _predict_trt(rgb_pil, prompt, mode='hybrid'):
    rgb_pil = rgb_pil.resize((ENG_IMG_W, ENG_IMG_H), _RESAMPLE)
    msg = [{'role':'user','content':[{'type':'image','image':rgb_pil},{'type':'text','text':prompt}]}]
    text = processor.py_apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    images, videos = processor.process_vision_info(msg)
    enc = processor(text=[text], images=images, videos=videos, return_tensors='pt')
    # Sanity-check: the engine was baked for (_grid_h, _grid_w); if the processor's
    # smart_resize produced anything else, fail loudly rather than reshape garbage.
    _gh_chk = enc['image_grid_hws']
    if not torch.is_tensor(_gh_chk):
        _gh_chk = torch.as_tensor(_gh_chk)
    _h_got, _w_got = int(_gh_chk[0,0]), int(_gh_chk[0,1])
    if (_h_got, _w_got) != (_grid_h, _grid_w):
        raise RuntimeError(
            f'processor produced grid_hws=({_h_got},{_w_got}) but the vision engine was '
            f'baked for ({_grid_h},{_grid_w}). The processor is still re-scaling; set '
            f'processor.image_processor.min_pixels and max_pixels to {ENG_IMG_W*ENG_IMG_H}.')
    t0 = time.time()
    _, t = trt_runner.generate(
        _to_np(enc['pixel_values'],   np.float16),
        _to_np(enc['image_grid_hws'], np.int32),
        _to_np(enc['input_ids'],      np.int64),
        max_new_tokens=160, generation_mode=mode,
        temperature=0.0, top_p=1.0, repetition_penalty=1.0,
    )
    dt_ms = (time.time() - t0) * 1000
    boxes = parse_boxes(tokenizer.decode(t, skip_special_tokens=False), *rgb_pil.size)
    # Scale boxes back to original frame coords downstream.
    return boxes, dt_ms

def _predict_pt(rgb_pil, prompt):
    if 'model' not in globals() or globals().get('model') is None:
        return None, None
    rgb_pil = rgb_pil.resize((ENG_IMG_W, ENG_IMG_H), _RESAMPLE)
    msg = [{'role':'user','content':[{'type':'image','image':rgb_pil},{'type':'text','text':prompt}]}]
    text = processor.py_apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    images, videos = processor.process_vision_info(msg)
    enc = processor(text=[text], images=images, videos=videos, return_tensors='pt').to('cuda')
    # Normalise image_grid_hws dtype (same hazard as cell 11).
    _gh_n = enc['image_grid_hws']
    enc['image_grid_hws'] = (_gh_n.to(dtype=torch.int32) if torch.is_tensor(_gh_n)
                              else torch.as_tensor(_gh_n, dtype=torch.int32, device=enc['pixel_values'].device))
    enc['pixel_values'] = enc['pixel_values'].to(REF_DTYPE)
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(
            pixel_values=enc['pixel_values'], input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'], image_grid_hws=enc['image_grid_hws'],
            tokenizer=tokenizer, max_new_tokens=160, use_cache=True,
            generation_mode='hybrid', temperature=0.0, do_sample=False, verbose=False,
        )
    dt_ms = (time.time() - t0) * 1000
    ot = out[0] if isinstance(out, tuple) else out
    if torch.is_tensor(ot):
        txt = tokenizer.decode(ot[0, enc['input_ids'].shape[1]:], skip_special_tokens=False)
    else:
        txt = str(ot)
    return parse_boxes(txt, *rgb_pil.size), dt_ms

# Active runtime registry
PATHS = [
    ('TRT + MTP',  lambda pil, prompt: _predict_trt(pil, prompt, mode='hybrid')),
    ('TRT (AR)',   lambda pil, prompt: _predict_trt(pil, prompt, mode='slow')),
]
if 'model' in globals() and globals().get('model') is not None:
    PATHS.append(('PyTorch',  lambda pil, prompt: _predict_pt(pil, prompt)))
print(f'Comparing {len(PATHS)} runtimes:', ', '.join(n for n,_ in PATHS))

def _draw_panel(bgr_frame, boxes_normed, label, ms):
    p = cv2.resize(bgr_frame, (PANEL_W, PANEL_H))
    Hf, Wf = bgr_frame.shape[:2]
    # boxes_normed are in the engine-resized image space (ENG_IMG_W x ENG_IMG_H).
    # Scale back to original frame, then to panel size.
    sx_panel, sy_panel = PANEL_W / Wf, PANEL_H / Hf
    sx_eng, sy_eng = Wf / ENG_IMG_W, Hf / ENG_IMG_H
    for (x1,y1,x2,y2) in boxes_normed:
        x1f, y1f = x1*sx_eng, y1*sy_eng
        x2f, y2f = x2*sx_eng, y2*sy_eng
        cv2.rectangle(p, (int(x1f*sx_panel), int(y1f*sy_panel)),
                         (int(x2f*sx_panel), int(y2f*sy_panel)), (0,255,0), 2)
    cv2.rectangle(p, (0,0), (PANEL_W, 36), (0,0,0), -1)
    cv2.putText(p, f'{label}  {ms:6.0f}ms  ({len(boxes_normed)} dets)',
                (10, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
    return p

# Process frames
if not (_uploaded and CUSTOM_VID.exists()):
    print('No custom video uploaded; skipping side-by-side render.')
else:
    CMP_OUT = WORK / 'demo_compare.mp4'
    cap = cv2.VideoCapture(str(CUSTOM_VID))
    fps_v = cap.get(cv2.CAP_PROP_FPS) or 24
    out_w, out_h = PANEL_W * len(PATHS), PANEL_H
    vw = cv2.VideoWriter(str(CMP_OUT), cv2.VideoWriter_fourcc(*'mp4v'), fps_v, (out_w, out_h))
    metrics = {name: {'lat_ms': [], 'n_dets': []} for name, _ in PATHS}
    agree_pairs = []
    i, t0 = 0, time.time()
    while i < MAX_COMPARE_FRAMES:
        ok, bgr = cap.read()
        if not ok: break
        rgb_pil = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        panels, boxes_per = [], []
        for name, fn in PATHS:
            try:
                boxes, ms = fn(rgb_pil, LUGGAGE_PROMPT)
                if boxes is None: raise RuntimeError('runtime returned None')
                metrics[name]['lat_ms'].append(ms)
                metrics[name]['n_dets'].append(len(boxes))
                boxes_per.append(boxes)
            except Exception as e:
                print(f'  frame {i} [{name}]: {e}')
                boxes, ms = [], 0.0
                boxes_per.append(boxes)
            panels.append(_draw_panel(bgr, boxes, name, ms))
        # IoU agreement between path 0 and path 1 (TRT+MTP vs TRT-AR by default)
        if len(boxes_per) >= 2 and boxes_per[0] and boxes_per[1]:
            ious_frame = [max((iou(b0, b1) for b1 in boxes_per[1]), default=0) for b0 in boxes_per[0]]
            if ious_frame: agree_pairs.append(float(np.mean(ious_frame)))
        composite = np.concatenate(panels, axis=1)
        vw.write(composite)
        i += 1
        if i % 5 == 0: print(f'  processed {i} frames ({(time.time()-t0)/i*1000:.0f} ms/frame avg)')
    vw.release(); cap.release()
    print(f'\nCompared {i} frames in {time.time()-t0:.1f}s → {CMP_OUT}')


### 17.2 · Side-by-side metrics + auto-download

The table below summarizes per-runtime latency, throughput, and detection density. The detection-agreement IoU (between TRT + MTP and TRT-AR) tells you whether the runtimes are landing boxes on the same objects — values above 0.7 mean they substantively agree on what's a bag and what isn't; lower values indicate the MTP path diverges enough to matter for your data distribution.


In [ ]:
if not (_uploaded and CUSTOM_VID.exists()) or 'metrics' not in globals():
    print('No metrics available — upload a video and re-run §17.1.')
else:
    print()
    print(f'{"Runtime":<14}  {"Avg lat":>10}  {"P50":>8}  {"P99":>8}  {"FPS":>6}  {"Dets/frame":>10}  {"Total dets":>10}')
    print('-' * 80)
    for name, m in metrics.items():
        if not m['lat_ms']:
            print(f'{name:<14}  {"skipped":>10}')
            continue
        lat = np.array(m['lat_ms']); nd = np.array(m['n_dets'])
        avg = lat.mean(); p50 = np.percentile(lat, 50); p99 = np.percentile(lat, 99)
        fps = 1000.0 / avg if avg > 0 else 0
        print(f'{name:<14}  {avg:7.1f} ms  {p50:5.1f} ms  {p99:5.1f} ms  {fps:5.2f}  {nd.mean():9.2f}  {int(nd.sum()):>10}')
    if agree_pairs:
        print()
        print(f'TRT+MTP vs TRT-AR detection agreement (mean per-frame IoU): {np.mean(agree_pairs):.3f}')
        print(f'  → agreement above 0.7 means the two runtimes mostly converge on the same bag locations')
        print(f'  → lower means MTP\'s parallel decoding shifts box coordinates enough to matter on your data')

    print()
    print(f'Side-by-side video saved to: {CMP_OUT}')
    print(f'  Inspect via the Colab Files panel, or auto-download below.')
    try:
        from google.colab import files as _gfiles
        _gfiles.download(str(CMP_OUT))
    except Exception:
        pass
